### Importación de librerías necesárias:

In [88]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import zipfile
import faiss
import pandas as pd
import numpy as np
from collections import Counter
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from typing import Literal
from pydantic import BaseModel


from dotenv import load_dotenv
from openai import OpenAI

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [3]:
DATA_DIR = Path("../Data/averitec")

RAW_DIR = DATA_DIR / "raw"
CLAIMS_DIR = RAW_DIR / "claims"
KNOWLEDGE_DIR = RAW_DIR / "knowledge_store"

PROCESSED_DIR = DATA_DIR / "processed"

CLAIMS_DIR.mkdir(parents=True, exist_ok=True)
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

### 1. Descargamos Claims:

In [4]:
averitec = load_dataset(
    "pminervini/averitec"
)

averitec

DatasetDict({
    train: Dataset({
        features: ['cached_original_claim_url', 'speaker', 'required_reannotation', 'reporting_source', 'label', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'claim', 'justification', 'location_ISO_code', 'original_claim_url', 'questions', 'claim_date'],
        num_rows: 3068
    })
    dev: Dataset({
        features: ['cached_original_claim_url', 'speaker', 'required_reannotation', 'reporting_source', 'label', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'claim', 'justification', 'location_ISO_code', 'original_claim_url', 'questions', 'claim_date'],
        num_rows: 500
    })
})

In [5]:
print(averitec["train"])
print(averitec["dev"])

Dataset({
    features: ['cached_original_claim_url', 'speaker', 'required_reannotation', 'reporting_source', 'label', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'claim', 'justification', 'location_ISO_code', 'original_claim_url', 'questions', 'claim_date'],
    num_rows: 3068
})
Dataset({
    features: ['cached_original_claim_url', 'speaker', 'required_reannotation', 'reporting_source', 'label', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'claim', 'justification', 'location_ISO_code', 'original_claim_url', 'questions', 'claim_date'],
    num_rows: 500
})


### Inspeccionando Claims

In [6]:
sample_claim = averitec["dev"][0]

# sample_claim
# sample_claim.keys()

### Guardando Claims - Averitec

In [7]:
train_df = averitec["train"].to_pandas()
dev_df = averitec["dev"].to_pandas()

In [7]:
len(train_df)
len(dev_df)

500

In [ ]:
train_df.to_json(
    CLAIMS_DIR / "train.json",
    orient="records",
    force_ascii=False,
    indent=2
)

dev_df.to_json(
    CLAIMS_DIR / "dev.json",
    orient="records",
    force_ascii=False,
    indent=2
)

### 2. Preparar AVeriTeC claims para retrieval experiments

In [8]:
train_sample = averitec["train"].select(range(5))

train_sample

Dataset({
    features: ['cached_original_claim_url', 'speaker', 'required_reannotation', 'reporting_source', 'label', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'claim', 'justification', 'location_ISO_code', 'original_claim_url', 'questions', 'claim_date'],
    num_rows: 5
})

In [9]:
for i, item in enumerate(train_sample):
    print(f"\n--- CLAIM {i} ---")
    print("Claim:", item["claim"])
    print("Date:", item["claim_date"])
    print("Speaker:", item["speaker"])
    print("Reporting source:", item["reporting_source"])
    print("Gold label:", item["label"])


--- CLAIM 0 ---
Claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.
Date: 25-8-2020
Speaker: Pam Bondi
Reporting source: Speech at The Republican National Convention
Gold label: Supported

--- CLAIM 1 ---
Claim: Donald Trump delivered the largest tax cuts in American history.
Date: 25-8-2020
Speaker: Eric Trump
Reporting source: Speech at The Republican National Convention
Gold label: Refuted

--- CLAIM 2 ---
Claim: In Nigeria … in terms of revenue share, 20% goes to the local government.
Date: 25-8-2020
Speaker: Raila Odinga
Reporting source: YouTube
Gold label: Supported

--- CLAIM 3 ---
Claim: Biden has pledged to stop border wall construction and give amnesty and health care to all illegal immigrants.
Date: 25-8-2020
Speaker: Eric Trump
Reporting source: Speech at The Republican National Convention
Gold label: Supported

--- CLAIM 4 ---
Claim: After the police shooting of Jacob Blake, Gov. Tony Evers & Lt. Gov. Mandela Barn

Aquí el Gold label lo mostramos solo para inspeccionar el dataset, no porque vaya a entrar al sistema.

### 3. Download AVeriTeC knowledge store

### Knowledge Store

In [9]:
AVerITEC_REPO = "chenxwh/AVeriTeC"

DEV_KNOWLEDGE_FILE = (
    "data_store/knowledge_store/"
    "dev_knowledge_store.zip"
)

In [10]:
repo_files = list_repo_files(
    repo_id="chenxwh/AVeriTeC"
)

knowledge_files = [
    f for f in repo_files
    if "knowledge_store" in f
]

print(len(knowledge_files))

for f in knowledge_files[:len(knowledge_files)]:
    print(f)

16
data_store/knowledge_store/dev_knowledge_store.zip
data_store/knowledge_store/test/test_0_499.zip
data_store/knowledge_store/test/test_1000_1499.zip
data_store/knowledge_store/test/test_1500_1999.zip
data_store/knowledge_store/test/test_2000_2214.zip
data_store/knowledge_store/test/test_500_999.zip
data_store/knowledge_store/test_updated/output_test_0_499.zip
data_store/knowledge_store/test_updated/output_test_1000_1499.zip
data_store/knowledge_store/test_updated/output_test_1500_1999.zip
data_store/knowledge_store/test_updated/output_test_2000_2214.zip
data_store/knowledge_store/test_updated/output_test_500_999.zip
data_store/knowledge_store/test_updated/readme.md
data_store/knowledge_store/train/train_0_999.zip
data_store/knowledge_store/train/train_1000_1999.zip
data_store/knowledge_store/train/train_2000_3067.zip
src/retrieval/scraper_for_knowledge_store.py


In [ ]:
# knowledge_zip = hf_hub_download(
#     repo_id="chenxwh/AVeriTeC",
#     filename="data_store/knowledge_store/dev_knowledge_store.zip",
#     local_dir=KNOWLEDGE_DIR
# )

### 4. Download a subset of the AVeriTeC train knowledge store

In [11]:
TRAIN_KNOWLEDGE_FILE = (
    "data_store/knowledge_store/train/"
    "train_0_999.zip"
)

knowledge_zip = hf_hub_download(
    repo_id="chenxwh/AVeriTeC",
    filename=TRAIN_KNOWLEDGE_FILE,
    local_dir=KNOWLEDGE_DIR
)

print(knowledge_zip)

C:\Users\natal\OneDrive\Documentos\UCM - Big data, ML and IA\MODULOS\TFM\TFM_Natalia_De_Oliveira_AgenteFakeNews\Data\averitec\raw\knowledge_store\data_store\knowledge_store\train\train_0_999.zip


In [12]:
knowledge_zip = Path(knowledge_zip)

print("Exists:", knowledge_zip.exists())
print("File:", knowledge_zip.name)
print(
    "Size MB:",
    round(knowledge_zip.stat().st_size / 1024**2, 2)
)

Exists: True
File: train_0_999.zip
Size MB: 19769.99


In [13]:
with zipfile.ZipFile(knowledge_zip, "r") as zip_ref:
    names = zip_ref.namelist()

print("Files in ZIP:", len(names))

for name in names[:len(names)]:
    print(name)

Files in ZIP: 1000
0.json
1.json
2.json
3.json
4.json
5.json
6.json
7.json
8.json
9.json
10.json
11.json
12.json
13.json
14.json
15.json
16.json
17.json
18.json
19.json
20.json
21.json
22.json
23.json
24.json
25.json
26.json
27.json
28.json
29.json
30.json
31.json
32.json
33.json
34.json
35.json
36.json
37.json
38.json
39.json
40.json
41.json
42.json
43.json
44.json
45.json
46.json
47.json
48.json
49.json
50.json
51.json
52.json
53.json
54.json
55.json
56.json
57.json
58.json
59.json
60.json
61.json
62.json
63.json
64.json
65.json
66.json
67.json
68.json
69.json
70.json
71.json
72.json
73.json
74.json
75.json
76.json
77.json
78.json
79.json
80.json
81.json
82.json
83.json
84.json
85.json
86.json
87.json
88.json
89.json
90.json
91.json
92.json
93.json
94.json
95.json
96.json
97.json
98.json
99.json
100.json
101.json
102.json
103.json
104.json
105.json
106.json
107.json
108.json
109.json
110.json
111.json
112.json
113.json
114.json
115.json
116.json
117.json
118.json
119.json
120.json
12

¿Como es cada json?

In [13]:
with zipfile.ZipFile(knowledge_zip, "r") as zip_ref:
    with zip_ref.open("0.json") as f:
        for i in range(3):
            line = f.readline()
            print(f"LINE {i}:")
            print(line[:500])
            print()

LINE 0:
b'{"claim_id": "0", "type": "gold", "query": "_", "url": "https://en.wikipedia.org/wiki/Hunter_Biden", "url2text": ["Robert Hunter Biden (born February 4, 1970) is an American attorney and businessman. Biden has also worked as a hedge fund principal and a venture capital and private equity fund investor.", "He formerly worked as a banker, a lobbyist, and a legal representative for lobbying firms.", "Biden is the second son of U.S. President Joe Biden and his first wife, Neilia Hunter Biden. In 197'

LINE 1:
b'{"claim_id": "0", "type": "question", "query": "Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014", "url": "https://www.reuters.com/article/idUSKBN1WX1P6/", "url2text": []}\n'

LINE 2:
b'{"claim_id": "0", "type": "question", "query": "Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014", "url": "https://apnews.com/artic

In [14]:
knowledge_records = []

with zipfile.ZipFile(knowledge_zip, "r") as zip_ref:
    with zip_ref.open("0.json") as f:
        for line in f:
            line = line.decode("utf-8").strip()

            if line:
                knowledge_records.append(
                    json.loads(line)
                )

print(type(knowledge_records))
print("Number of records:", len(knowledge_records))

<class 'list'>
Number of records: 546


In [ ]:
knowledge_records[0]

{'claim_id': '0',
 'type': 'gold',
 'query': '_',
 'url': 'https://en.wikipedia.org/wiki/Hunter_Biden',
 'url2text': ['Robert Hunter Biden (born February 4, 1970) is an American attorney and businessman. Biden has also worked as a hedge fund principal and a venture capital and private equity fund investor.',
  'He formerly worked as a banker, a lobbyist, and a legal representative for lobbying firms.',
  'Biden is the second son of U.S. President Joe Biden and his first wife, Neilia Hunter Biden. In 1972, when Biden was two years old, a car crash killed his mother, who was driving, and his one-year-old sister, Naomi, and seriously injured both him and his older brother, Beau.',
  "In his memoir, Beautiful Things, Biden wrote of his struggles with drug and alcohol abuse, which escalated after Beau's 2015 death from brain cancer.[1][2]",
  'He was discharged from the U.S. Navy Reserve shortly after his commissioning, due to a failed drug test.',
  'Biden was a founding board member of BH

In [15]:
with_text = [
    r for r in knowledge_records
    if r.get("url2text")
]

without_text = [
    r for r in knowledge_records
    if not r.get("url2text")
]

print("With text:", len(with_text))
print("Without text:", len(without_text))

With text: 431
Without text: 115


In [16]:
type_counts = Counter(
    r.get("type")
    for r in knowledge_records
)

type_counts

Counter({'question_duplicate': 86,
         'provenance': 85,
         'gpt_url_only': 75,
         'gpt_question': 49,
         'NER': 48,
         'background_questions': 44,
         'most_similar': 44,
         'background': 35,
         'question': 33,
         'same_entity_questions': 19,
         'answer': 17,
         'claim+question': 7,
         'claim': 3,
         'gold': 1})

`0.json` no contiene únicamente “documentos web”

In [17]:
print("Number of records:", len(knowledge_records))
print("With text:", len(with_text))
print("Without text:", len(without_text))
print(type_counts)

Number of records: 546
With text: 431
Without text: 115
Counter({'question_duplicate': 86, 'provenance': 85, 'gpt_url_only': 75, 'gpt_question': 49, 'NER': 48, 'background_questions': 44, 'most_similar': 44, 'background': 35, 'question': 33, 'same_entity_questions': 19, 'answer': 17, 'claim+question': 7, 'claim': 3, 'gold': 1})


No vamos a trabajar con los 546 registros directamente. Nos conviene filtrar registros sin texto, posibles duplicados y registros con type = `gold`.

In [18]:
retrieval_records = [
    r for r in knowledge_records
    if r.get("url2text")
    # and r.get("type") != "gold"
]

print(
    "Records for retrieval:",
    len(retrieval_records)
)

Records for retrieval: 431


In [19]:
urls = [
    r.get("url")
    for r in retrieval_records
    if r.get("url")
]

print("Total URLs:", len(urls))
print("Unique URLs:", len(set(urls)))

Total URLs: 431
Unique URLs: 431


In [20]:
url_counts = Counter(urls)

duplicate_urls = {
    url: count
    for url, count in url_counts.items()
    if count > 1
}

print(
    "Duplicated URLs:",
    len(duplicate_urls)
)

Duplicated URLs: 0


### 5. Creando documentos para la fase RAG

In [ ]:
def clean_text_parts(text_parts):
    """
    Limpia los fragmentos de texto extraídos de una fuente.

    Elimina valores no textuales, strings vacíos y fragmentos exactamente duplicados, manteniendo el orden 
    original.

    Parameters
    ----------
    text_parts : list
        Lista de fragmentos de texto.

    Returns
    -------
    list
        Lista de fragmentos limpios y sin duplicados exactos.
    """
    cleaned_parts = []
    seen = set()

    for part in text_parts:
        if not isinstance(part, str):
            continue

        part = part.strip()

        if not part:
            continue

        # Evita fragmentos exactamente repetidos
        if part in seen:
            continue

        seen.add(part)
        cleaned_parts.append(part)

    return cleaned_parts

In [ ]:
def build_document(record):
    """
    Convierte un registro del Knowledge Store de AVeriTeC en una estructura documental utilizable por el 
    pipeline.

    Parameters
    ----------
    record : dict
        Registro original del Knowledge Store.

    Returns
    -------
    dict
        Documento normalizado con texto y metadatos.
    """
    text_parts = clean_text_parts(
        record.get("url2text", [])
    )

    text = " ".join(text_parts)

    return {
        "claim_id": record.get("claim_id"),
        "url": record.get("url"),
        "source_type": record.get("type"),
        "query": record.get("query"),
        "text_parts": text_parts,
        "text": text
    }

In [19]:
documents = [
    build_document(record)
    for record in retrieval_records
]

print("Documents:", len(documents))

Documents: 431


In [24]:
documents[0]

{'claim_id': '0',
 'url': 'https://en.wikipedia.org/wiki/Hunter_Biden',
 'source_type': 'gold',
 'query': '_',
 'text_parts': ['Robert Hunter Biden (born February 4, 1970) is an American attorney and businessman. Biden has also worked as a hedge fund principal and a venture capital and private equity fund investor.',
  'He formerly worked as a banker, a lobbyist, and a legal representative for lobbying firms.',
  'Biden is the second son of U.S. President Joe Biden and his first wife, Neilia Hunter Biden. In 1972, when Biden was two years old, a car crash killed his mother, who was driving, and his one-year-old sister, Naomi, and seriously injured both him and his older brother, Beau.',
  "In his memoir, Beautiful Things, Biden wrote of his struggles with drug and alcohol abuse, which escalated after Beau's 2015 death from brain cancer.[1][2]",
  'He was discharged from the U.S. Navy Reserve shortly after his commissioning, due to a failed drug test.',
  'Biden was a founding board mem

Antes de obtener los chuncks para cada documento, necesitamos saber más o menos el tamaño:

In [25]:
document_lengths = [
    len(doc["text"])
    for doc in documents
]

print("Min:", min(document_lengths))
print("Max:", max(document_lengths))
print(
    "Average:",
    sum(document_lengths) / len(document_lengths)
)

Min: 71
Max: 3601339
Average: 60943.7030162413


In [26]:
lengths_df = pd.Series(
    document_lengths
)

lengths_df.describe()

count    4.310000e+02
mean     6.094370e+04
std      2.884391e+05
min      7.100000e+01
25%      2.881500e+03
50%      5.078000e+03
75%      1.088100e+04
max      3.601339e+06
dtype: float64

Ahora mismo tenemos, 1 URL - 1 Documento, y lo que queremos tener es 1 Documentos - Varios chunks

`Las 10 URLs/documentos completos más grandes dentro de los 430 documentos asociados a la claim 0.`

In [27]:
largest_documents = sorted(
    documents,
    key=lambda x: len(x["text"]),
    reverse=True
)[:10]

for i, doc in enumerate(largest_documents):
    print(f"\n--- DOCUMENT {i} ---")
    print("Length:", len(doc["text"]))
    print("URL:", doc["url"])
    print("Type:", doc["source_type"])
    print("Number of text parts:", len(doc["text_parts"]))


--- DOCUMENT 0 ---
Length: 3601339
URL: https://healthmasters.com/daily-news-articles-archive
Type: most_similar
Number of text parts: 42458

--- DOCUMENT 1 ---
Length: 2365109
URL: https://www.nps.gov/aboutus/foia/upload/COVID-19-Emails-Correspondence-DOI-NPS-2020-000736-Release-Date-01032022.pdf
Type: question_duplicate
Number of text parts: 35069

--- DOCUMENT 2 ---
Length: 2018490
URL: https://www.govinfo.gov/content/pkg/CHRG-116hhrg39405/pdf/CHRG-116hhrg39405.pdf
Type: most_similar
Number of text parts: 24250

--- DOCUMENT 3 ---
Length: 1955363
URL: https://www.intelligence.senate.gov/sites/default/files/documents/report_volume5.pdf
Type: answer
Number of text parts: 21893

--- DOCUMENT 4 ---
Length: 1695500
URL: https://www.congress.gov/116/crpt/hrpt346/CRPT-116hrpt346.pdf
Type: gpt_question
Number of text parts: 25747

--- DOCUMENT 5 ---
Length: 1534213
URL: https://docs.house.gov/billsthisweek/20191216/CRPT-116hrpt346.pdf
Type: gpt_question
Number of text parts: 16930

--- DOC

In [28]:
for i, doc in enumerate(largest_documents[:5]):
    print(f"\n--- DOCUMENT {i} PREVIEW ---")
    print(doc["text"][:1000])


--- DOCUMENT 0 PREVIEW ---
Bidenomics Fail: White House Plans Downshift In Electric Vehicle Transition As Demand Slides Rolling Disaster: Ford Halts 2024 F-150 Lightning Shipments Doctor warns WHO pandemic treaty includes 'gain of function' data sharing WHO pandemic treaty: “Torrent of fake news” has put negotiations at risk, says WHO chief Midwest and East Coast set to see the mercury soar 20 to 40 degrees above average as February ends with record-smashing warmth as low-pressure system sweeps across the U.S. Cruise ship 'hit by CHOLERA outbreak': Brits among thousands of passengers stuck in quarantine on Norwegian Dawn liner 'floating aimlessly' off coast of Africa after it was barred from docking in Mauritius 'to avoid health risks' Politico Reporter on MSNBC Frets That Christian Nationalists Believe Americans' Rights Come From God, Not the Government (VIDEO) “Waves of refugees in their millions will be unleashed…reaching far into…the USA”: Billy Meier, 1947 Banks will 'tokenize' c

### 6. Chunks

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

In [30]:
chunks = []

for doc_id, doc in enumerate(documents):
    split_texts = text_splitter.split_text(
        doc["text"]
    )

    for chunk_id, chunk_text in enumerate(split_texts):
        chunks.append({
            "claim_id": doc["claim_id"],
            "document_id": doc_id,
            "chunk_id": chunk_id,
            "url": doc["url"],
            "source_type": doc["source_type"],
            "query": doc["query"],
            "text": chunk_text
        })

In [ ]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 431
Chunks: 31051


Cada chunks conserva correctamente claim_id, document_id, url, source_type, query y el texto del chunk.

In [ ]:
chunks[0]

{'claim_id': '0',
 'document_id': 0,
 'chunk_id': 0,
 'url': 'https://en.wikipedia.org/wiki/Hunter_Biden',
 'source_type': 'gold',
 'query': '_',
 'text': "Robert Hunter Biden (born February 4, 1970) is an American attorney and businessman. Biden has also worked as a hedge fund principal and a venture capital and private equity fund investor. He formerly worked as a banker, a lobbyist, and a legal representative for lobbying firms. Biden is the second son of U.S. President Joe Biden and his first wife, Neilia Hunter Biden. In 1972, when Biden was two years old, a car crash killed his mother, who was driving, and his one-year-old sister, Naomi, and seriously injured both him and his older brother, Beau. In his memoir, Beautiful Things, Biden wrote of his struggles with drug and alcohol abuse, which escalated after Beau's 2015 death from brain cancer.[1][2] He was discharged from the U.S. Navy Reserve shortly after his commissioning, due to a failed drug test. Biden was a founding board 

¿Cuantos chunks genera cada URL?

In [ ]:
chunks_per_document = Counter(
    chunk["document_id"]
    for chunk in chunks
)

pd.Series(
    list(chunks_per_document.values())
).describe()

count     431.000000
mean       72.044084
std       339.319093
min         1.000000
25%         4.000000
50%         6.000000
75%        13.000000
max      4237.000000
dtype: float64

- Tenemos 431 documentos;
- Cada documento genera en promedio ~72 chunks;
- El documento típico genera bastante menos: la mediana es ~6;
- El 75% genera 13 chunks o menos;
- Un documento extremo genera 4.237 chunks.

In [ ]:
print(
    "Maximum chunks from one document:",
    max(chunks_per_document.values())
)

Maximum chunks from one document: 4237


In [ ]:
sum(chunks_per_document.values()) #Número total de chunks

31051

El chunking funciona técnicamente, pero todavía no pasamos a embeddings sin controlar el efecto de los documentos extremadamente grandes.

In [ ]:
chunk_counts = pd.Series(
    chunks_per_document
).sort_values(ascending=False)

chunk_counts.head(10)

321    4237
29     2783
335    2374
91     2301
132    1995
133    1806
341    1102
46     1078
144     912
349     851
dtype: int64

¿Qué porcentaje del vector store acabaría perteneciendo solo a los 10 documentos más grandes?

In [ ]:
top_10_chunks = chunk_counts.head(10).sum()

print(
    "Chunks from top 10 largest documents:",
    top_10_chunks
)

print(
    "Percentage of all chunks:",
    round(
        100 * top_10_chunks / len(chunks),
        2
    )
)

Chunks from top 10 largest documents: 19439
Percentage of all chunks: 62.6


Los diez documentos que generan la mayor cantidad de chunks representan el 62,6% de todos los chunks para el claim 0. Esto refleja la presencia de varias páginas web y documentos PDF muy grandes en la base de conocimientos de AVeriTeC. Estos documentos no se eliminarán porque podrían contener evidencia relevante; en su lugar, la recuperación incluirá más adelante una restricción de diversidad de fuentes para evitar que un solo documento domine el conjunto final de evidencia.

Todos los chunks participarán en la búsqueda y cualquiera podrá ser recuperado. Lo que limitaremos es cuántos fragmentos de una misma fuente llegarán al conjunto final de evidencias.

### 7. Embeddings

¿Que queremos saber? ¿qué vectores de chunks están más cerca del vector de la claim?

In [ ]:
len(chunks)

31051

In [21]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2303.64it/s]


Aún no embebemos los 30961 chunks, probamos apenas con uno.

In [ ]:
sample_text = chunks[0]["text"]

sample_embedding = embedding_model.encode(
    sample_text,
    normalize_embeddings=True
)

print(type(sample_embedding))
print(sample_embedding.shape) #Las 384 dimensiones conjuntamente representan la información semántica del texto.

<class 'numpy.ndarray'>
(384,)


Embedding del claim 0:

In [ ]:
claim_0 = train_df.iloc[0]["claim"]

In [ ]:
claim_0

'Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.'

In [ ]:
claim_embedding = embedding_model.encode(
    claim_0,
    normalize_embeddings=True
)

print(claim_embedding.shape)

(384,)


Vamos a hacer una similitud manual.

In [ ]:
similarity = np.dot(
    claim_embedding,
    sample_embedding
)

print(similarity) #Cuanto mayor sea, más similitud semántica.
#No es probabilidad de que el chunk demuestre que la claim es verdadera. Solo significa: este fragmento parece 
#semánticamente relacionado con la claim.

# Un chunk puede tener una similitud altísima y estar refutando la claim.
# Eso lo veremos verificado después el Evidence Verifier.

0.6494279


In [ ]:
print(sample_embedding.shape)
print(claim_embedding.shape)
print(similarity)

(384,)
(384,)
0.6494279


Generando embeddings completos:

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

In [ ]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 486/486 [2:04:23<00:00, 15.36s/it]  


In [ ]:
print(type(chunk_embeddings))
print(chunk_embeddings.shape) #30.961 vectores de 384 dimensiones

<class 'numpy.ndarray'>
(31051, 384)


### 8. Vector store

En este paso, crearemos la estructura que permite buscar eficientemente qué vectores son más parecidos a una query (claim).

In [ ]:
embedding_dim = chunk_embeddings.shape[1]

In [ ]:
index = faiss.IndexFlatIP(embedding_dim)

In [ ]:
print(index.ntotal) #tenemos que añadir los embeddings

0


In [ ]:
chunk_embeddings_faiss = (chunk_embeddings.astype("float32"))

In [ ]:
index.add(chunk_embeddings_faiss)

In [ ]:
print("Vectores indexados:",index.ntotal)

Vectores indexados: 31051


In [ ]:
query_embedding = (
    claim_embedding
    .astype("float32")
    .reshape(1, -1)
)

In [ ]:
query_embedding.shape

(1, 384)

### 9. Retrieval

Recuperamos los 10 más proximos:

In [ ]:
k = 10
scores, indices = index.search(
    query_embedding,
    k
)

In [ ]:
print(scores.shape)
print(indices.shape)

(1, 10)
(1, 10)


In [ ]:
print(scores[0]) #score es similitud semántica.
print(indices[0])

[0.8068609  0.79276973 0.7905766  0.75146747 0.74707484 0.74565995
 0.73625815 0.7356435  0.7335564  0.73089916]
[ 3058  5644  3059  8344  5908   153   163  5915 23765 30980]


El chunk chunks[3058] es uno de los fragmentos con mayor similitud semántica respecto al claim 0.

In [ ]:
retrieved_chunks = []

for score, idx in zip(
    scores[0],
    indices[0]
):
    chunk = chunks[int(idx)]

    retrieved_chunks.append({
        "score": float(score),
        "document_id": chunk["document_id"],
        "chunk_id": chunk["chunk_id"],
        "url": chunk["url"],
        "source_type": chunk["source_type"],
        "text": chunk["text"]
    })

El claim que estamos buscando es: `Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma`

In [ ]:
for i, result in enumerate(
    retrieved_chunks,
    start=1
):
    print(f"\n--- RESULT {i} ---")
    print("Score:", result["score"])
    print(
        "Document:",
        result["document_id"]
    )
    print(
        "Chunk:",
        result["chunk_id"]
    )
    print("URL:", result["url"])
    print(
        "Text:",
        result["text"][:700]
    )


--- RESULT 1 ---
Score: 0.8068609237670898
Document: 30
Chunk: 0
URL: https://www.washingtonexaminer.com/?p=731178
Text: A defense attorney for President Trump’s Senate impeachment trial slammed Hunter Biden’s employment with Ukrainian gas company Burisma Holdings, saying the former vice president’s son lacked the experience necessary to land the job and its salary. “Hunter Biden is paid over $83,000 a month, while the average American family of four during that time, each year, made less than $54,000, and that’s according to U.S. Census Bureau during that time,” said Pam Bondi. Bondi, 54, submitted a list of articles she said shows Biden did not have the background required to sit on the board of Burisma. She suggested it was his connection to his father that helped secure the lucrative gig and shield him fro

--- RESULT 2 ---
Score: 0.792769730091095
Document: 61
Chunk: 1
URL: https://www.cnn.com/politics/live-news/impeachment-hearing-11-19-19/h_e2df04c755f6130b6b48459d8a6d6171
Text

Los primeros resultados contienen precisamente conceptos como Hunter Biden, Burisma, Ukraine, energy sector, experience, natural gas, etc. No está recuperando simplemente documentos sobre política estadounidense en general.

Según los documentos recuperados, en la búsqueda de los 10 más cercanos a nuestra claim, el retriever está encontrando exactamente la información que se esperaba, pero aqui todavía no nos dice: SUPPORTED O REFUTED

En este ejemplo concreto, la concentración de chunks no está provocando que una sola URL domine la búsqueda, como nos temíamos.

El retrieval semántico basado en embeddings recupera fragmentos altamente relacionados con la claim de consulta. En la primera prueba, 9 de los 10 resultados proceden de documentos distintos, por lo que la fuerte concentración de chunks observada en algunos documentos extensos no ha provocado, en este caso, una monopolización evidente de los resultados.Pero `atención`, aqui tenemos una sola claim, no podemos todavía afirmar que el retriever funciona perfectamente.

### 10. Retrieval Evaluation usando Averitec (Train)

Vamos a hacer una comprobación intermedia, usando la clasificación que ya proporciona averitec

In [ ]:
gold_questions = train_df.iloc[0]["questions"]

for i, question in enumerate(gold_questions):
    print(f"\n--- GOLD QUESTION {i} ---")
    print("Question:", question["question"])

    for answer in question["answers"]:
        print("Answer:", answer["answer"])
        print("Source URL:", answer["source_url"])


--- GOLD QUESTION 0 ---
Question: Did Hunter Biden have any experience in the energy sector at the time he joined the board of the  Burisma energy company in 2014
Answer: No
Source URL: https://en.wikipedia.org/wiki/Hunter_Biden

--- GOLD QUESTION 1 ---
Question: Did Hunter Biden have any experience in Ukraine at the time he joined the board of the  Burisma energy company in 2014
Answer: No
Source URL: https://en.wikipedia.org/wiki/Hunter_Biden


Aunque nuestro retrieval no haya devuelto Wikipedia en el Top-10, sí ha encontrado otro documento que contiene la información necesaria para verificar las dos cuestiones.

¿Cuál es el mejor chunk de la fuente gold y en qué posición global estaría?

In [ ]:
gold_url = "https://en.wikipedia.org/wiki/Hunter_Biden"

gold_indices = [
    i
    for i, chunk in enumerate(chunks)
    if chunk["url"] == gold_url
]

print("Number of gold chunks:", len(gold_indices))

Number of gold chunks: 90


In [ ]:
gold_scores = np.array([
    np.dot(
        query_embedding[0],
        chunk_embeddings_faiss[i]
    )
    for i in gold_indices
])

best_gold_position = np.argmax(gold_scores)

best_gold_idx = gold_indices[best_gold_position]
best_gold_score = gold_scores[best_gold_position]

print("Best gold chunk index:", best_gold_idx)
print("Best gold score:", best_gold_score)
print()
print(chunks[best_gold_idx]["text"][:1000])

Best gold chunk index: 13
Best gold score: 0.7020207

an attorney with Boies Schiller Flexner, and a consulting firm in which Biden is a partner was also retained by Burisma.[65][66][67] Christopher Heinz, John Kerry's stepson, opposed his partners Devon Archer and Hunter Biden joining the board in 2014 due to the reputational risk.[63] Biden served on the board of Burisma until his term expired in April 2019,[66] receiving compensation of up to $50,000 per month in some months.[66][65] Because Joe Biden played a major role in U.S. policy towards Ukraine, some Ukrainian anti-corruption advocates[68][69] and Obama administration officials expressed concern that Hunter Biden having joined the board could create the appearance of a conflict of interest and undermine Joe Biden's anti-corruption work in Ukraine.[19][63] While serving as vice president, Joe Biden joined other Western leaders in encouraging the government of Ukraine to fire the country's top prosecutor Viktor Shokin,[70][71] 

La evidencia gold existe y es bastante relevante, pero hay 44 chunks que el modelo considera semánticamente más parecidos.

In [ ]:
all_scores = np.dot(
    chunk_embeddings_faiss,
    query_embedding[0]
)

gold_rank = (
    np.sum(all_scores > best_gold_score)
    + 1
)

print("Best gold chunk rank:", gold_rank)

Best gold chunk rank: 45


El mejor chunk perteneciente a la fuente gold de Wikipedia ocupa el rank 45 dentro de todos los chunks ordenados por similitud con la claim 0.

### 11. Funciones reutilizables y empaquetado de funciones para el Pipeline final

A partir de este punto se agrupan las versiones definitivas de las funciones desarrolladas durante las etapas anteriores para poder reutilizarlas en el pipeline completo y además empaquetamos las piezas usadas individualmente como funciones definitivas.

In [22]:
def clean_text_parts(text_parts):
    cleaned_parts = []
    seen = set()

    for part in text_parts:
        if not isinstance(part, str):
            continue

        part = part.strip()

        if not part:
            continue

        # Evita fragmentos exactamente repetidos
        if part in seen:
            continue

        seen.add(part)
        cleaned_parts.append(part)

    return cleaned_parts

In [23]:
def build_document(record):
    text_parts = clean_text_parts(
        record.get("url2text", [])
    )

    text = " ".join(text_parts)

    return {
        "claim_id": record.get("claim_id"),
        "url": record.get("url"),
        "source_type": record.get("type"),
        "query": record.get("query"),
        "text_parts": text_parts,
        "text": text
    }

In [24]:
def load_knowledge_records(knowledge_zip, claim_id):
    """
    Carga los registros del Knowledge Store asociados a una claim de AVeriTeC.

    La función localiza el archivo JSON correspondiente a la claim aunque se encuentre dentro de una 
    subcarpeta del ZIP, permitiendo utilizar la misma función con los Knowledge Stores de train y dev.

    Parameters
    ----------
    knowledge_zip : Path
        Ruta al archivo ZIP del Knowledge Store.

    claim_id : int
        Identificador de la claim.

    Returns
    -------
    list
        Registros del Knowledge Store asociados a la claim.
    """

    target_file = f"{claim_id}.json"

    knowledge_records = []

    with zipfile.ZipFile(knowledge_zip, "r") as zip_ref:

        # Buscamos el JSON de la claim independientemente de la carpeta interna del ZIP.
        matching_files = [
            name
            for name in zip_ref.namelist()
            if name.rsplit("/", 1)[-1] == target_file
        ]

        if not matching_files:
            raise FileNotFoundError(
                f"No se encontró {target_file} dentro de {knowledge_zip}"
            )

        file_name = matching_files[0]

        with zip_ref.open(file_name) as f:

            for line in f:

                line = line.decode("utf-8").strip()

                if line:
                    knowledge_records.append(
                        json.loads(line)
                    )

    return knowledge_records

In [ ]:
def retrieve_evidence(
    claim,
    embedding_model,
    index,
    chunks,
    candidate_k=50,
    final_k=10,
    max_chunks_per_document=2,
):
    """
    Recupera evidencias semánticamente relevantes para una afirmación, aplicando además una restricción de
    diversidad por documento.

    Parameters
    ----------
    claim : str
        Afirmación que se quiere verificar.

    embedding_model
        Modelo de SentenceTransformer utilizado para generar el embedding de la afirmación.

    index
        Índice FAISS que contiene los embeddings de los chunks.

    chunks : list[dict]
        Lista de chunks con su texto y metadatos.
        El orden de esta lista debe coincidir con el orden de los vectores almacenados en el índice FAISS.

    candidate_k : int, default=50
        Número de candidatos iniciales recuperados mediante FAISS antes de aplicar la restricción de 
        diversidad.

    final_k : int, default=10
        Número máximo de chunks de evidencia que se devolverán finalmente.

    max_chunks_per_document : int, default=2
        Número máximo de chunks que se permite seleccionar procedentes del mismo documento.

    Returns
    -------
    list[dict]
        Lista de evidencias recuperadas. Para cada chunk se devuelve su score de similitud, identificador 
        del documento, identificador del chunk, URL, tipo de fuente y texto.
    """

    # 1. Generamos el embedding de la afirmación
    query_embedding = embedding_model.encode(
        claim,
        normalize_embeddings=True
    )

    # FAISS espera una matriz bidimensional:
    # (número de consultas, dimensión del embedding)
    query_embedding = (
        query_embedding
        .astype("float32")
        .reshape(1, -1)
    )

    # 2. Recuperamos un conjunto inicial amplio de candidatos
    # ordenados por similitud semántica con la afirmación
    scores, indices = index.search(
        query_embedding,
        candidate_k
    )

    # 3. Seleccionamos las evidencias finales aplicando
    # diversidad por documento
    selected = []

    # Guarda cuántos chunks hemos seleccionado de cada documento
    document_counts = {}

    for score, idx in zip(scores[0], indices[0]):

        # FAISS puede devolver -1 si no encuentra suficientes resultados
        if idx == -1:
            continue

        chunk = chunks[int(idx)]

        document_id = chunk["document_id"]

        # Número de chunks ya seleccionados de este documento
        current_count = document_counts.get(
            document_id,
            0
        )

        # Si ya hemos alcanzado el máximo permitido para este documento, ignoramos este chunk
        if current_count >= max_chunks_per_document:
            continue

        # Añadimos el chunk al conjunto final de evidencias
        selected.append({
            "score": float(score),
            "document_id": document_id,
            "chunk_id": chunk["chunk_id"],
            "url": chunk["url"],
            "source_type": chunk["source_type"],
            "text": chunk["text"],
        })

        # Actualizamos el contador de chunks seleccionados para este documento
        document_counts[document_id] = (
            current_count + 1
        )

        # Terminamos cuando alcanzamos el número deseado de evidencias finales
        if len(selected) == final_k:
            break

    return selected

In [26]:
def select_relevant_documents(
    claim,
    documents,
    top_n=100,
):
    """
    Preselecciona los documentos más relacionados con una afirmación mediante similitud TF-IDF.

    Esta etapa se utiliza como filtro previo al chunking y a la generación de embeddings, con el objetivo de reducir el 
    coste computacional del retrieval semántico.

    Parameters
    ----------
    claim : str
        Afirmación para la que se quieren recuperar evidencias.

    documents : list[dict]
        Lista completa de documentos asociados al knowledge store
        de la claim.

    top_n : int, default=100
        Número máximo de documentos que se conservarán después
        de la preselección.

    Returns
    -------
    list[dict]
        Documentos seleccionados y ordenados de mayor a menor
        similitud TF-IDF con la afirmación.
    """

    document_texts = [
        document["text"]
        for document in documents
    ]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=30000
    )

    # Representamos la claim y los documentos en el mismo
    # espacio TF-IDF
    tfidf_matrix = vectorizer.fit_transform(
        [claim] + document_texts
    )

    claim_vector = tfidf_matrix[0]
    document_vectors = tfidf_matrix[1:]

    similarities = cosine_similarity(
        claim_vector,
        document_vectors
    )[0]

    ranked_indices = similarities.argsort()[::-1]

    selected_documents = []

    for idx in ranked_indices[:top_n]:

        document = documents[int(idx)].copy()

        document["document_score"] = float(
            similarities[idx]
        )

        selected_documents.append(document)

    return selected_documents

In [ ]:
def deduplicate_documents_by_url(documents):
    """
    Elimina documentos duplicados que apuntan a la misma URL.

    Se conserva únicamente la primera aparición de cada URL.

    Parameters
    ----------
    documents : list
        Lista de documentos normalizados.

    Returns
    -------
    list
        Lista de documentos sin URLs duplicadas.
    """
    documents_by_url = {}

    for document in documents:
        url = document["url"]

        if url not in documents_by_url:
            documents_by_url[url] = document
            continue

        current_document = documents_by_url[url]

        if len(document["text"]) > len(current_document["text"]):
            documents_by_url[url] = document

    return list(documents_by_url.values())

In [ ]:
def build_claim_index(
    claim,
    knowledge_records,
    embedding_model,
    text_splitter,
    max_chunks_before_filter=40000,
    top_n_documents=100,
):
    """
    Construye el índice vectorial de una claim de AVeriTeC.

    El proceso:
    1. transforma los registros del Knowledge Store en documentos;
    2. elimina documentos vacíos;
    3. elimina URLs duplicadas;
    4. realiza un chunking inicial;
    5. aplica una preselección TF-IDF si el número de chunks
       supera el umbral definido;
    6. genera los chunks definitivos;
    7. calcula embeddings;
    8. construye un índice FAISS.

    Parameters
    ----------
    claim : str
        Afirmación para la que se quiere construir el índice.

    knowledge_records : list[dict]
        Registros del knowledge store asociados a la claim.

    embedding_model
        Modelo SentenceTransformer utilizado para generar embeddings.

    text_splitter
        Objeto utilizado para dividir los documentos en chunks.

    max_chunks_before_filter : int, default=40000
        Número máximo de chunks que se permite procesar directamente.
        Si se supera este valor, se aplica una preselección TF-IDF de documentos.

    top_n_documents : int, default=100
        Número de documentos que se conservarán cuando sea necesario aplicar la preselección TF-IDF.

    Returns
    -------
    chunks : list[dict]
        Lista final de chunks utilizados para construir el índice.

    index
        Índice FAISS que contiene los embeddings de los chunks.

    info : dict
        Información sobre el proceso de construcción del índice, incluyendo número de documentos, número de chunks y si
        se utilizó preselección TF-IDF.
    """

    # 1. Conservamos únicamente los registros que contienen texto
    retrieval_records = [
        record
        for record in knowledge_records
        if record.get("url2text")
    ]

    # 2. Construimos los documentos
    documents = [
        build_document(record)
        for record in retrieval_records
    ]

    original_documents = len(documents)

    # 3. Eliminamos URLs duplicadas
    documents = deduplicate_documents_by_url(
        documents
    )

    unique_documents = len(documents)

    # 4. Generamos inicialmente los chunks para conocer el tamaño real del corpus
    chunks = []

    for document_id, document in enumerate(documents):

        split_texts = text_splitter.split_text(
            document["text"]
        )

        for chunk_id, chunk_text in enumerate(split_texts):

            chunks.append({
                "claim_id": document["claim_id"],
                "document_id": document_id,
                "chunk_id": chunk_id,
                "url": document["url"],
                "source_type": document["source_type"],
                "query": document["query"],
                "text": chunk_text,
            })

    initial_chunks = len(chunks)

    # 5. Comprobamos si el corpus es demasiado grande
    use_tfidf_filter = (
        initial_chunks > max_chunks_before_filter
    )

    if use_tfidf_filter:

        print(
            f"Corpus grande detectado: {initial_chunks} chunks."
        )

        print(
            f"Se aplicará TF-IDF para seleccionar "
            f"{top_n_documents} documentos."
        )

        # Seleccionamos los documentos más relacionados
        documents = select_relevant_documents(
            claim=claim,
            documents=documents,
            top_n=top_n_documents
        )

        # Volvemos a crear los chunks,
        # esta vez únicamente con los documentos seleccionados
        chunks = []

        for document_id, document in enumerate(documents):

            split_texts = text_splitter.split_text(
                document["text"]
            )

            for chunk_id, chunk_text in enumerate(split_texts):

                chunks.append({
                    "claim_id": document["claim_id"],
                    "document_id": document_id,
                    "chunk_id": chunk_id,
                    "url": document["url"],
                    "source_type": document["source_type"],
                    "query": document["query"],
                    "text": chunk_text,
                })

    else:

        print(
            f"Corpus manejable: {initial_chunks} chunks."
        )

        print(
            "No es necesario aplicar preselección TF-IDF."
        )

    final_chunks = len(chunks)

    print("Documentos originales:", original_documents)
    print("Documentos únicos:", unique_documents)
    print("Documentos finales:", len(documents))
    print("Chunks iniciales:", initial_chunks)
    print("Chunks finales:", final_chunks)

    # 6. Extraemos únicamente el texto de los chunks
    chunk_texts = [
        chunk["text"]
        for chunk in chunks
    ]

    # 7. Generamos los embeddings SOLO del corpus final
    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    chunk_embeddings = chunk_embeddings.astype(
        "float32"
    )

    # 8. Construimos el índice FAISS
    embedding_dim = chunk_embeddings.shape[1]

    index = faiss.IndexFlatIP(
        embedding_dim
    )

    index.add(
        chunk_embeddings
    )

    # Información útil para analizar posteriormente el proceso
    info = {
        "original_documents": original_documents,
        "unique_documents": unique_documents,
        "final_documents": len(documents),
        "initial_chunks": initial_chunks,
        "final_chunks": final_chunks,
        "tfidf_filter_used": use_tfidf_filter,
    }

    return chunks, index, info

Probamos las funciones:

In [39]:
knowledge_records_1 = load_knowledge_records(
    knowledge_zip,
    claim_id=1
)

claim_1 = train_df.iloc[1]["claim"]

chunks_1, index_1, info_1 = build_claim_index(
    claim=claim_1,
    knowledge_records=knowledge_records_1,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    max_chunks_before_filter=40000,
    top_n_documents=100,
)

Corpus grande detectado: 69700 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 768
Documentos únicos: 767
Documentos finales: 100
Chunks iniciales: 69700
Chunks finales: 1845


Batches: 100%|██████████| 29/29 [03:28<00:00,  7.20s/it]


In [44]:
results = retrieve_evidence(
    claim=claim_1,
    embedding_model=embedding_model,
    index=index_1,
    chunks=chunks_1,
    candidate_k=50,
    final_k=10,
    max_chunks_per_document=2,
)

In [ ]:
for i, result in enumerate(results, start=1):

    print(f"\n--- RESULT {i} ---")
    print("Score:", round(result["score"],3))
    print("Document:", result["document_id"])
    print("Chunk:", result["chunk_id"])
    print("URL:", result["url"])
    print("Text:", result["text"][:700])


--- RESULT 1 ---
Score: 0.876
Document: 1
Chunk: 0
URL: https://trumpwhitehouse.archives.gov/briefings-statements/president-donald-j-trump-achieved-biggest-tax-cuts-reforms-american-history/
Text: BIGGEST TAX CUTS AND REFORMS: Because of President Donald J. Trump, Americans will benefit from the biggest tax cuts and reforms in American history. - President Trump’s tax cuts are the biggest gross tax cuts in American history, cutting over $5.5 trillion in taxes over ten years. - The President’s tax law included substantial reforms to make taxes simpler and fairer, which helped offset the cost of the tax cuts and thereby limit the net tax cut to $1.5 trillion. - President Trump cut the corporate tax rate from 35 percent to 21 percent, the largest percentage point reduction of the top marginal rate in history. - President Trump’s tax cuts include the biggest increase in the child tax cred

--- RESULT 2 ---
Score: 0.845
Document: 11
Chunk: 0
URL: https://www.theguardian.com/us-news/2017/ap

In [ ]:
gold_questions_1 = train_df.iloc[1]["questions"]

for i, question in enumerate(gold_questions_1):
    print(f"\n--- GOLD QUESTION {i} ---")
    print("Question:", question["question"])

    for answer in question["answers"]:
        print("Answer:", answer["answer"])
        print("Source URL:", answer["source_url"])


--- GOLD QUESTION 0 ---
Question: Did the 2017 tax bill deliver the largest tax cuts in American history?
Answer: This tax cut is the 8th largest as a percent of Gross Domestic Product (GDP) since 1918 and the 4th largest in inflation-adjusted dollars.
Source URL: https://www.crfb.org/blogs/president-trumps-tax-cut-largest-history-yet

--- GOLD QUESTION 1 ---
Question: Has there been larger tax bills than the 2017 tax bill?
Answer: Three tax bills have been larger: American Taxpayer Relief Act of 2012 (enacted in 2013); 
Tax Relief, Unemployment Insurance Reauthorization, and Job Creation Act of 2010 and
Economic Recovery Tax Act of 1981
Source URL: https://www.treasury.gov/resource-center/tax-policy/tax-analysis/Documents/WP81-Table2013.pdf


observandos las URL Gold

In [ ]:
gold_urls_1 = set()

for question in gold_questions_1:
    for answer in question["answers"]:
        if answer["source_url"]:
            gold_urls_1.add(answer["source_url"])

print(gold_urls_1)

{'https://www.crfb.org/blogs/president-trumps-tax-cut-largest-history-yet', 'https://www.treasury.gov/resource-center/tax-policy/tax-analysis/Documents/WP81-Table2013.pdf'}


In [ ]:
retrieved_corpus_urls = {
    chunk["url"]
    for chunk in chunks_1
}

for url in gold_urls_1:
    print(
        url,
        "->",
        "PRESENTE" if url in retrieved_corpus_urls else "NO PRESENTE"
    )

https://www.crfb.org/blogs/president-trumps-tax-cut-largest-history-yet -> PRESENTE
https://www.treasury.gov/resource-center/tax-policy/tax-analysis/Documents/WP81-Table2013.pdf -> NO PRESENTE


Con top_n_documents=100, una de las dos fuentes gold de la claim 1 no quedó en el corpus preseleccionado, aunque el retrieval final recuperó múltiples evidencias alternativas pertinentes.

### 12. Evidence Verifier

In [29]:
class EvidenceAssessment(BaseModel):
    """
    Evaluación de una evidencia concreta respecto a la claim.
    """

    evidence_id: int

    relation: Literal[
        "SUPPORTS",
        "REFUTES",
        "NEUTRAL"
    ]

    reason: str


class VerificationResult(BaseModel):
    """
    Resultado global de la verificación factual de una claim utilizando exclusivamente las evidencias 
    recuperadas.
    """

    verdict: Literal[
        "SUPPORTED",
        "REFUTED",
        "NOT_ENOUGH_EVIDENCE",
        "CONFLICTING_EVIDENCE"
    ]

    evidence_sufficient: bool

    evidence_assessments: list[EvidenceAssessment]

    explanation: str

In [89]:
env_path = Path("../.env")
loaded = load_dotenv(dotenv_path=env_path)
client = OpenAI()

Transformando Results al formato deseado para el LLM:

In [ ]:
def format_evidences_for_llm(evidences):
    """
    Convierte las evidencias recuperadas por el RAG en un texto structurado que pueda ser utilizado por el 
    Evidence Verifier.

    No se incluyen los scores de similitud para evitar que el LLM interprete relevancia semántica como grado
    de veracidad.

    Parameters
    ----------
    evidences : list[dict]
        Lista de evidencias devuelta por retrieve_evidence().

    Returns
    -------
    str
        Evidencias numeradas con su URL y texto.
    """

    formatted_evidences = []

    for evidence_id, evidence in enumerate(evidences, start=1):

        formatted_evidences.append(
            f"""
            EVIDENCE {evidence_id}
            URL: {evidence["url"]}
            TEXT:
            {evidence["text"]}
            """.strip()
            )

    return "\n\n".join(formatted_evidences)

Probamos la función:

In [ ]:
# evidence_text.__class__

In [45]:
evidence_text = format_evidences_for_llm(results)

print(evidence_text)

EVIDENCE 1
            URL: https://trumpwhitehouse.archives.gov/briefings-statements/president-donald-j-trump-achieved-biggest-tax-cuts-reforms-american-history/
            TEXT:
            BIGGEST TAX CUTS AND REFORMS: Because of President Donald J. Trump, Americans will benefit from the biggest tax cuts and reforms in American history. - President Trump’s tax cuts are the biggest gross tax cuts in American history, cutting over $5.5 trillion in taxes over ten years. - The President’s tax law included substantial reforms to make taxes simpler and fairer, which helped offset the cost of the tax cuts and thereby limit the net tax cut to $1.5 trillion. - President Trump cut the corporate tax rate from 35 percent to 21 percent, the largest percentage point reduction of the top marginal rate in history. - President Trump’s tax cuts include the biggest increase in the child tax credit in history. - The Child Tax Credit was increased by $573.4 billion over ten years, larger than any previ

In [ ]:
def verify_evidence(
    claim,
    evidences,
    client,
    model="gpt-5.6-terra",
    language=None,
):
    """
    Verifica una claim utilizando exclusivamente las evidencias recuperadas por el sistema RAG.

    Parameters
    ----------
    claim : str
        Afirmación que se quiere verificar.

    evidences : list[dict]
        Evidencias recuperadas por retrieve_evidence().

    client
        Cliente de OpenAI.

    model : str
        Modelo LLM utilizado para realizar la verificación.

    Returns
    -------
    VerificationResult
        Resultado estructurado de la verificación factual.
    """

    # Preparamos las evidencias recuperadas en un formato comprensible para el LLM
    evidence_text = format_evidences_for_llm(evidences)

    if language == "es":
        language_instruction = (
            "Write the explanation and reasons in Spanish."
        )
    elif language == "en":
        language_instruction = (
            "Write the explanation and reasons in English."
        )
    
    instructions = f"""
    You are an Evidence Verifier for a fact-checking system.

    Your task is to verify a claim using ONLY the evidence provided to you.

    Do not use external knowledge.
    Do not assume facts that are not present in the evidence.

    A claim may contain multiple factual components.
    When this happens, consider the different components of the claim before assigning evidence relationships and the 
    final verdict.

    For each evidence item, determine its relationship to the claim:

    - SUPPORTS:
    The evidence provides factual information that supports the claim as a whole, or supports an important component without
    contradicting or leaving unresolved another essential component of the claim.

    - REFUTES:
    The evidence provides factual information that contradicts the claim as a whole, or directly contradicts an essential 
    component in a way that makes the overall claim false.

    - NEUTRAL:
    The evidence is related to the claim but does not provide enough information to support or refute it as a whole. 
    Use NEUTRAL when the evidence addresses only one component of a multi-part claim and leaves other essential
    components unresolved.

    Then determine the overall verdict:

    - SUPPORTED:
    The available evidence is sufficient to support the claim as a whole.

    - REFUTED:
    The available evidence is sufficient to refute the claim as a whole.

    - NOT_ENOUGH_EVIDENCE:
    The available evidence does not contain enough information to reach a justified conclusion about the claim.

    - CONFLICTING_EVIDENCE:
    The available evidence supports some important factual aspects of the claim while refuting, contradicting, or 
    substantially qualifying other important aspects, so that labeling the whole claim simply SUPPORTED or REFUTED would
    lose relevant information.

    Use this verdict also for cherry-picking cases, where the claim relies on factually supported information but 
    selectively combines, omits, or frames evidence in a way that produces a misleading overall conclusion.

    Do not automatically label a multi-part claim REFUTED simply because one component is contradicted if other important 
    components are supported.
    In such cases, consider whether CONFLICTING_EVIDENCE is more appropriate.

    Set evidence_sufficient to true when the provided evidence contains enough factual information to justify the selected
    verdict, including when the evidence is sufficient to establish that the case is CONFLICTING_EVIDENCE.

    Set evidence_sufficient to false when the appropriate verdict is
    NOT_ENOUGH_EVIDENCE.

    Assess every evidence item provided.

    Base the final verdict on the content of the evidence, not on retrieval similarity scores.

    {language_instruction}
    """

    response = client.responses.parse(
        model=model,
        instructions=instructions,
        input=f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {evidence_text}
    """,
            text_format=VerificationResult,
        )

    return response.output_parsed

Probamos la función verify_evidence para la claim 1:

In [ ]:
verification_1 = verify_evidence(
    claim=claim_1,
    evidences=results,
    client=client,
)

print(verification_1.model_dump_json(indent=2))

{
  "verdict": "REFUTED",
  "evidence_sufficient": true,
  "evidence_assessments": [
    {
      "evidence_id": 1,
      "relation": "SUPPORTS",
      "reason": "The archived Trump White House statement explicitly says that Trump's tax cuts were the biggest gross tax cuts and reforms in American history, citing more than $5.5 trillion in gross cuts over ten years."
    },
    {
      "evidence_id": 2,
      "relation": "NEUTRAL",
      "reason": "This article reports that the Trump administration called its proposal the biggest tax cut in history, but it describes an announced proposal rather than establishing that Trump delivered the largest enacted tax cuts. It also does not independently verify the historical comparison."
    },
    {
      "evidence_id": 3,
      "relation": "REFUTES",
      "reason": "CNN states that the enacted tax cuts would not be the biggest in history under the government's way of measuring tax cuts, identifying Reagan's cuts as definitely larger and Kennedy/

In [ ]:
gold_label_1 = train_df.iloc[1]["label"]

print("Predicted verdict:", verification_1.verdict)
print("Gold label:", gold_label_1)

Predicted verdict: REFUTED
Gold label: Refuted


### 12.1 Función global

In [32]:
def verify_averitec_claim(
    claim_id,
    claim,
    knowledge_zip,
    embedding_model,
    text_splitter,
    client,
):
    """
    Ejecuta el pipeline completo de verificación factual para una claim de AVeriTeC.

    Parameters
    ----------
    claim_id : int
        Identificador de la claim en AVeriTeC.

    claim : str
        Texto de la afirmación que se quiere verificar.

    knowledge_zip : str or Path
        ZIP que contiene el knowledge store correspondiente.

    embedding_model
        Modelo utilizado para generar embeddings.

    text_splitter
        Objeto utilizado para dividir los documentos en chunks.

    client
        Cliente de OpenAI utilizado por el Evidence Verifier.

    Returns
    -------
    dict
        Resultado completo del pipeline, incluyendo información del retrieval, evidencias recuperadas y
        verificación factual.
    """

    # 1. Cargamos el knowledge store asociado a la claim
    knowledge_records = load_knowledge_records(
        knowledge_zip=knowledge_zip,
        claim_id=claim_id
    )

    # 2. Construimos el índice vectorial
    chunks, index, retrieval_info = build_claim_index(
        claim=claim,
        knowledge_records=knowledge_records,
        embedding_model=embedding_model,
        text_splitter=text_splitter
    )

    # 3. Recuperamos las evidencias más relevantes
    evidences = retrieve_evidence(
        claim=claim,
        embedding_model=embedding_model,
        index=index,
        chunks=chunks
    )

    # 4. El LLM verifica la claim utilizando esas evidencias
    verification = verify_evidence(
        claim=claim,
        evidences=evidences,
        client=client
    )

    return {
        "claim_id": claim_id,
        "claim": claim,
        "retrieval_info": retrieval_info,
        "evidences": evidences,
        "verification": verification,
    }

Los labels definidos en AveriTec son: 'Supported', 'Refuted', 'Conflicting Evidence/Cherrypicking','Not Enough Evidence' y el LLM nos da: SUPPORTED, REFUTED, NOT_ENOUGH_EVIDENCE, CONFLICTING_EVIDENCE, por lo tanto creamos una función para normalizar los labels.

In [33]:
def normalize_averitec_label(label):
    """
    Convierte las etiquetas originales de AVeriTeC al formato
    utilizado por el Evidence Verifier.
    """

    label_mapping = {
        "Supported": "SUPPORTED",
        "Refuted": "REFUTED",
        "Not Enough Evidence": "NOT_ENOUGH_EVIDENCE",
        "Conflicting Evidence/Cherrypicking": "CONFLICTING_EVIDENCE",
    }

    return label_mapping[label]

Antes de probar el pipeline completo, comprobamos la salida con las labels normalizadas:

In [ ]:
gold_label_1 = train_df.iloc[1]["label"]

gold_normalized_1 = normalize_averitec_label(
    gold_label_1
)

print("Predicted:", verification_1.verdict)
print("Gold:", gold_normalized_1)
print(
    "Correct:",
    verification_1.verdict == gold_normalized_1
)

Antes de dar continuidad a probar el pipeline para todos los claims, vamos a comprobar con Claim 0:

In [45]:
claim_id_test = 0

claim_test = train_df.iloc[claim_id_test]["claim"]

result_0 = verify_averitec_claim(
    claim_id=claim_id_test,
    claim=claim_test,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus manejable: 31051 chunks.
No es necesario aplicar preselección TF-IDF.
Documentos originales: 431
Documentos únicos: 431
Documentos finales: 431
Chunks iniciales: 31051
Chunks finales: 31051


Batches: 100%|██████████| 486/486 [1:41:08<00:00, 12.49s/it]  


In [46]:
predicted_0 = result_0["verification"].verdict

gold_0 = normalize_averitec_label( #Normalizamos la label de averitec con el LLM
    train_df.iloc[claim_id_test]["label"]
)

print("Claim:", claim_test)
print("Predicted:", predicted_0)
print("Gold:", gold_0)
print("Correct:", predicted_0 == gold_0)

Claim: Hunter Biden had no experience in Ukraine or in the energy sector when he joined the board of Burisma.
Predicted: SUPPORTED
Gold: SUPPORTED
Correct: True


In [47]:
#Viendo el resultado:
print(
    result_0["verification"].model_dump_json(indent=2)
)

{
  "verdict": "SUPPORTED",
  "evidence_sufficient": true,
  "evidence_assessments": [
    {
      "evidence_id": 1,
      "relation": "SUPPORTS",
      "reason": "It reports Pam Bondi's statement that Hunter Biden had no experience in natural gas, the energy sector, or Ukrainian regulatory affairs before joining Burisma."
    },
    {
      "evidence_id": 2,
      "relation": "SUPPORTS",
      "reason": "It states that Republicans pointed out Hunter Biden had no experience in the energy sector before taking the Burisma job. It does not address Ukraine-specific experience."
    },
    {
      "evidence_id": 3,
      "relation": "SUPPORTS",
      "reason": "It reports the statement that Hunter Biden had no experience in natural gas, the energy sector, or Ukrainian regulatory affairs, and identifies his Burisma board service as beginning in 2014."
    },
    {
      "evidence_id": 4,
      "relation": "SUPPORTS",
      "reason": "It explicitly describes Hunter Biden as 'a man with no exp

Hasta aqui tenemos una señal muy importante, porque ahora tenemos dos casos distintos funcionando end-to-end: claim 0 y claim 1, pero todavía no podemos hablar de rendimiento del sistema: son solo dos casos.

Vamos a probar con una claim con una label un poco más complicada `Not Enough Evidence`. 

In [52]:
nei_sample = train_df[
    train_df["label"] == "Not Enough Evidence"
].iloc[0]

claim_id_nei = nei_sample.name
claim_nei = nei_sample["claim"]

print("Claim ID:", claim_id_nei)
print("Claim:", claim_nei)
print("Gold:", nei_sample["label"])

Claim ID: 21
Claim: There is a ‘global average’ for the number of judges and magistrates to number of people in Kenya.
Gold: Not Enough Evidence


In [53]:
result_nei = verify_averitec_claim(
    claim_id=claim_id_nei,
    claim=claim_nei,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 106806 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 780
Documentos únicos: 780
Documentos finales: 100
Chunks iniciales: 106806
Chunks finales: 6051


Batches: 100%|██████████| 95/95 [20:06<00:00, 12.70s/it]


In [54]:
predicted_nei = result_nei["verification"].verdict

gold_nei = normalize_averitec_label(
    nei_sample["label"]
)

print("Claim:", claim_nei)
print("Predicted:", predicted_nei)
print("Gold:", gold_nei)
print("Correct:", predicted_nei == gold_nei)

Claim: There is a ‘global average’ for the number of judges and magistrates to number of people in Kenya.
Predicted: NOT_ENOUGH_EVIDENCE
Gold: NOT_ENOUGH_EVIDENCE
Correct: True


In [55]:
print(
    "Evidence sufficient:",
    result_nei["verification"].evidence_sufficient
)

Evidence sufficient: False


Vamos a probar la ultima label: `Conflicting Evidence/Cherrypicking` 

In [56]:
conflicting_sample = train_df[
    train_df["label"] == "Conflicting Evidence/Cherrypicking"
].iloc[0]

claim_id_conflicting = conflicting_sample.name
claim_conflicting = conflicting_sample["claim"]

print("Claim ID:", claim_id_conflicting)
print("Claim:", claim_conflicting)
print("Gold:", conflicting_sample["label"])

Claim ID: 7
Claim: Margaret Sanger was a racist who believed in eugenics. Her goal when founding Planned Parenthood was to eradicate minorities.
Gold: Conflicting Evidence/Cherrypicking


In [57]:
result_conflicting = verify_averitec_claim(
    claim_id=claim_id_conflicting,
    claim=claim_conflicting,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 93797 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 769
Documentos únicos: 767
Documentos finales: 100
Chunks iniciales: 93797
Chunks finales: 2033


Batches: 100%|██████████| 32/32 [05:27<00:00, 10.23s/it]


In [58]:
predicted_conflicting = (
    result_conflicting["verification"].verdict
)

gold_conflicting = normalize_averitec_label(
    conflicting_sample["label"]
)

print("Claim:", claim_conflicting)
print("Predicted:", predicted_conflicting)
print("Gold:", gold_conflicting)
print(
    "Correct:",
    predicted_conflicting == gold_conflicting
)

print(
    "Evidence sufficient:",
    result_conflicting["verification"].evidence_sufficient
)

Claim: Margaret Sanger was a racist who believed in eugenics. Her goal when founding Planned Parenthood was to eradicate minorities.
Predicted: REFUTED
Gold: CONFLICTING_EVIDENCE
Correct: False
Evidence sufficient: True


In [59]:
print(
    result_conflicting["verification"].model_dump_json(indent=2)
)

{
  "verdict": "REFUTED",
  "evidence_sufficient": true,
  "evidence_assessments": [
    {
      "evidence_id": 1,
      "relation": "SUPPORTS",
      "reason": "It describes Sanger's eugenics program, including her call for sterilization of people characterized as mentally or physically disabled, and states that Planned Parenthood cited her eugenicism and racism."
    },
    {
      "evidence_id": 2,
      "relation": "SUPPORTS",
      "reason": "It reports that Planned Parenthood of Greater New York cited Sanger's documented racist legacy and deep belief in eugenic ideology."
    },
    {
      "evidence_id": 3,
      "relation": "SUPPORTS",
      "reason": "It asserts that Sanger repeatedly expressed racist intentions and that Planned Parenthood manifested those ideologies. It does not specifically document an objective of eradicating minorities."
    },
    {
      "evidence_id": 4,
      "relation": "SUPPORTS",
      "reason": "This is substantively the same as Evidence 3 and asse

In [60]:
print("Gold justification:")
print(conflicting_sample["justification"])

Gold justification:
Evidence is conflicting with Margaret Sanger, as she was a progressive who believed in racial integration. She voted for Norman Thomas. She worked with progressive Black people—W.E.B. Du Bois, for example, who, along with Mary McCleod Bethune and Adam Clayton Powell Sr., served on the board of the Negro Project, a network of birth control and maternal health clinics Sanger established in Harlem and the South, etc. However, it does seem that Sanger believed in eugenics and thought people, especially poor people, often had too many kids to care for properly and that too many of those kids were born physically disabled (or, in the language of the day, "feeble-minded"). She did not oppose forced sterilization.


In [61]:
print("\nGold questions:")
print(conflicting_sample["questions"])


Gold questions:
[{'answers': array([{'answer': 'Eugenics is the practice or advocacy of improving the human species by selectively mating people with specific desirable hereditary traits. It aims to reduce human suffering by “breeding out” disease, disabilities and so-called undesirable characteristics from the human population', 'answer_type': 'Extractive', 'boolean_explanation': None, 'cached_source_url': 'https://web.archive.org/web/20230420103906/https://www.history.com/topics/european-history/eugenics', 'source_medium': 'Web text', 'source_url': 'https://www.history.com/topics/germany/eugenics'}],
       dtype=object), 'question': 'What is eugenics?'}
 {'answers': array([{'answer': "I think the greatest sin in the world is bringing children into the world -- that have disease from their parents, that have no chance in the world to be a human being practically. Delinquents, prisoners, all sorts of things just marked when they're born. That to me is the greatest sin -- that people 

El resultado no es el esperado y posiblemente algunas de las partes del pipeline necesite algunos ajustes, El primer paso realizado es modificar el prompt de la función `verify_evidence`, volver a ejecutar la función, sin generar nuevamente embeddings ni faiss, y ver que resultado aporta:

Old prompt:

instructions = <br>
"""
    You are an Evidence Verifier for a fact-checking system. <br>
    Your task is to verify a claim using ONLY the evidence provided to you. <br>
    Do not use external knowledge. <br>
    Do not assume facts that are not present in the evidence. <br>
    For each evidence item, determine its relationship to the claim: <br>
    - SUPPORTS: <br>
    The evidence contains information that supports the claim. <br>
    - REFUTES:<br>
    The evidence contains information that contradicts the claim.<br>
    - NEUTRAL:<br>
    The evidence is related to the topic but does not provide enough<br>
    information to support or refute the claim.<br>
    Then determine the overall verdict:<br>
    - SUPPORTED:<br>
    The available evidence is sufficient to support the claim.<br>
    - REFUTED:<br>
    The available evidence is sufficient to refute the claim.<br>
    - NOT_ENOUGH_EVIDENCE:<br>
    The evidence does not contain enough information to reach a conclusion.<br>
    - CONFLICTING_EVIDENCE:<br>
    The evidence contains substantial incompatible information supporting
    and refuting the claim, and the conflict cannot be resolved from the
    provided evidence. <br>
    Set evidence_sufficient to true only when the retrieved evidence contains
    enough factual information to reach a justified conclusion. <br>
    Assess every evidence item provided.<br>
    Base the final verdict on the content of the evidence, not on retrieval
    similarity scores.<br>
    Write all reasons and the final explanation in English.
"""

In [64]:
verification_conflicting_v2 = verify_evidence(
    claim=claim_conflicting,
    evidences=result_conflicting["evidences"],
    client=client
)

print(
    verification_conflicting_v2.model_dump_json(indent=2)
)

{
  "verdict": "CONFLICTING_EVIDENCE",
  "evidence_sufficient": true,
  "evidence_assessments": [
    {
      "evidence_id": 1,
      "relation": "NEUTRAL",
      "reason": "It characterizes Sanger's eugenics as racist and coercive and reports her support for sterilizing people with disabilities, supporting the eugenics/racism portion. However, it does not establish that her goal in founding Planned Parenthood was to eradicate minorities."
    },
    {
      "evidence_id": 2,
      "relation": "NEUTRAL",
      "reason": "It reports that Planned Parenthood of Greater New York cited Sanger's documented racist legacy and deep belief in eugenics. It does not provide evidence that Planned Parenthood was founded with the goal of eradicating minorities."
    },
    {
      "evidence_id": 3,
      "relation": "NEUTRAL",
      "reason": "It asserts that Sanger had racist intentions and that Planned Parenthood manifested those ideologies, but the excerpt provides no specific substantiation of th

Antes de dar al nuevo prompt como correcto, vamos a comprobar que no ha roto el razonamiento de las otras labels que antes salían bien:

In [66]:
#Claim 0 
verification_0_v2 = verify_evidence(
    claim=claim_test,
    evidences=result_0["evidences"],
    client=client
)

print("Predicted:", verification_0_v2.verdict)
print("Gold: SUPPORTED")
print(
    "Evidence sufficient:",
    verification_0_v2.evidence_sufficient
)

Predicted: SUPPORTED
Gold: SUPPORTED
Evidence sufficient: True


In [ ]:
#Not enough evidence
verification_nei_v2 = verify_evidence(
    claim=claim_nei,
    evidences=result_nei["evidences"],
    client=client
)

print("Predicted:", verification_nei_v2.verdict)
print("Gold: NOT_ENOUGH_EVIDENCE")
print(
    "Evidence sufficient:",
    verification_nei_v2.evidence_sufficient
)

Predicted: NOT_ENOUGH_EVIDENCE
Gold: NOT_ENOUGH_EVIDENCE
Evidence sufficient: False


In [75]:
#claim 1 
verification_1_v2 = verify_evidence(
    claim=claim_1,
    evidences=results,
    client=client
)

print("Predicted:", verification_1_v2.verdict)
print("Gold: REFUTED")
print(
    "Evidence sufficient:",
    verification_1_v2.evidence_sufficient
)

Predicted: REFUTED
Gold: REFUTED
Evidence sufficient: True


El diseño y el prompt V2 funcionan de forma coherente en las cuatro pruebas manuales realizadas.

Antes de asumir el nuevo prompt como el definitivo vamos a probar el pipeline completo con otras claims diferentes.  <br>

Primero seleccionamos una muestra pequeña y balanceada, por ejemplo 3 claims por clase = 12 claims, además de las que ya hemos usado. Así comprobamos que el pipeline aguanta varios ejemplos. <br>

Como ahora nuestro knowledge store descargado corresponde al bloque 0–999, podemos seleccionar únicamente claims de ese rango:

In [49]:
evaluation_set = train_df.loc[:999].copy()

evaluation_sample = (
    evaluation_set
    .groupby("label", group_keys=False)
    .sample(n=3, random_state=42) #random_state para la reproducibilidad
)

evaluation_sample[
    ["claim", "label"]
]

,claim,label
275,A study finds a diet heavy with kimchi—spiced ...,Conflicting Evidence/Cherrypicking
961,Michael Bloomberg “did not poll well as mayor ...,Conflicting Evidence/Cherrypicking
929,Government of India is monitoring all forms of...,Conflicting Evidence/Cherrypicking
777,Billionaire philanthropist and Microsoft found...,Not Enough Evidence
770,Half of UK adults are now being paid by the st...,Not Enough Evidence
255,"Sanitation coverage is now at 25%, from 16%.",Not Enough Evidence
233,"BLM injures 1000 police officers, kills36 peop...",Refuted
349,The CDC may have to stop calling COVID-19 an ‘...,Refuted
574,"Walmart Tweeted ""We wont be Rebuilding Walmart...",Refuted
611,The COVID-19 pandemic has helped bolster the r...,Supported


In [50]:
evaluation_sample.label.unique()

array(['Conflicting Evidence/Cherrypicking', 'Not Enough Evidence',
       'Refuted', 'Supported'], dtype=object)

En este caso no vamos a hacer claim por claim, vamos a crear una función que haga todo el proceso.

In [34]:
def evaluate_averitec_sample(
    evaluation_sample,
    knowledge_zip,
    embedding_model,
    text_splitter,
    client,
    output_path,
):
    """
    Evalúa una muestra de claims de AVeriTeC utilizando el pipeline completo de verificación factual.

    Los resultados se guardan después de evaluar cada claim.
    Si existe un archivo previo de resultados, la evaluación continúa únicamente con las claims pendientes.

    Parameters
    ----------
    evaluation_sample : pd.DataFrame
        Muestra de claims que se quiere evaluar.

    knowledge_zip : str or Path
        ZIP que contiene los knowledge stores.

    embedding_model
        Modelo utilizado para generar embeddings.

    text_splitter
        Divisor utilizado para generar chunks.

    client
        Cliente de OpenAI utilizado por el Evidence Verifier.

    output_path : str or Path
        Ruta donde se guardarán progresivamente los resultados.

    Returns
    -------
    pd.DataFrame
        Tabla con los resultados de evaluación.
    """

    output_path = Path(output_path)

    # 1. Comprobamos si ya existen resultados anteriores
    if output_path.exists():

        existing_results = pd.read_csv(
            output_path
        )

        evaluation_results = (
            existing_results.to_dict("records")
        )

        completed_claim_ids = set(
            existing_results["claim_id"]
            .astype(int)
            .tolist()
        )

        print(
            f"Resultados anteriores encontrados: "
            f"{len(completed_claim_ids)} claims."
        )

    else:

        evaluation_results = []
        completed_claim_ids = set()

    # 2. Recorremos la muestra de evaluación
    for claim_id, row in evaluation_sample.iterrows():

        claim_id = int(claim_id)

        # Si ya está evaluada, no repetimos el cálculo
        if claim_id in completed_claim_ids:

            print(
                f"\nClaim {claim_id} ya evaluada. "
                "Se omite."
            )

            continue

        claim = row["claim"]

        gold_label = normalize_averitec_label(
            row["label"]
        )

        print(f"\nEvaluando claim {claim_id}...")
        print("Gold:", gold_label)

        # 3. Ejecutamos el pipeline completo
        result = verify_averitec_claim(
            claim_id=claim_id,
            claim=claim,
            knowledge_zip=knowledge_zip,
            embedding_model=embedding_model,
            text_splitter=text_splitter,
            client=client,
        )

        predicted_label = (
            result["verification"].verdict
        )

        # 4. Guardamos el resultado de la claim
        evaluation_results.append({
            "claim_id": claim_id,
            "claim": claim,
            "gold": gold_label,
            "predicted": predicted_label,
            "correct": predicted_label == gold_label,
            "evidence_sufficient": (
                result["verification"].evidence_sufficient
            ),
            "explanation": (
                result["verification"].explanation
            ),
        })

        completed_claim_ids.add(claim_id)

        # 5. Guardamos un checkpoint después de cada claim
        results_df = pd.DataFrame(
            evaluation_results
        )

        results_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8"
        )

        print("Predicted:", predicted_label)
        print(
            "Correct:",
            predicted_label == gold_label
        )
        print(
            f"Resultados guardados: {len(results_df)}"
        )

    return pd.DataFrame(evaluation_results)

Vamos a crear un proceso de protección, en el caso de alguna interrupción en alguna de las 12 claims, tener las demás guardadas y no perder el proceso. Además, si tenemos que volver a ejecutar, la función detectará si la claim ya se ha guardado y no volverá a ejecutarla.

In [52]:
EVALUATION_RESULTS_PATH = (
    PROCESSED_DIR / "evaluation_sample_results.csv"
)

In [55]:
evaluation_results_df = evaluate_averitec_sample(
    evaluation_sample=evaluation_sample,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
    output_path=EVALUATION_RESULTS_PATH,
)

Resultados anteriores encontrados: 11 claims.

Claim 275 ya evaluada. Se omite.

Claim 961 ya evaluada. Se omite.

Claim 929 ya evaluada. Se omite.

Claim 777 ya evaluada. Se omite.

Claim 770 ya evaluada. Se omite.

Claim 255 ya evaluada. Se omite.

Claim 233 ya evaluada. Se omite.

Claim 349 ya evaluada. Se omite.

Claim 574 ya evaluada. Se omite.

Claim 611 ya evaluada. Se omite.

Claim 82 ya evaluada. Se omite.

Evaluando claim 845...
Gold: SUPPORTED
Corpus manejable: 38173 chunks.
No es necesario aplicar preselección TF-IDF.
Documentos originales: 623
Documentos únicos: 623
Documentos finales: 623
Chunks iniciales: 38173
Chunks finales: 38173


Batches: 100%|██████████| 597/597 [1:25:54<00:00,  8.63s/it]


Predicted: CONFLICTING_EVIDENCE
Correct: False
Resultados guardados: 12


In [56]:
evaluation_results_df[
    [
        "claim_id",
        "gold",
        "predicted",
        "correct",
        "evidence_sufficient"
    ]
]

,claim_id,gold,predicted,correct,evidence_sufficient
0,275,CONFLICTING_EVIDENCE,CONFLICTING_EVIDENCE,True,True
1,961,CONFLICTING_EVIDENCE,CONFLICTING_EVIDENCE,True,True
2,929,CONFLICTING_EVIDENCE,SUPPORTED,False,True
3,777,NOT_ENOUGH_EVIDENCE,REFUTED,False,True
4,770,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,False
5,255,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,False
6,233,REFUTED,NOT_ENOUGH_EVIDENCE,False,False
7,349,REFUTED,REFUTED,True,True
8,574,REFUTED,NOT_ENOUGH_EVIDENCE,False,False
9,611,SUPPORTED,SUPPORTED,True,True


### Análisis preliminar de la evaluación

Una vez validado manualmente el comportamiento del `Evidence Verifier` sobre ejemplos
representativos de las cuatro clases de AVeriTeC, se realizó una primera evaluación
exploratoria sobre una muestra balanceada de 12 afirmaciones del conjunto de entrenamiento,
seleccionando tres ejemplos por clase.

El sistema clasificó correctamente 7 de las 12 afirmaciones evaluadas. Este resultado no
se interpreta como una estimación final del rendimiento del sistema, ya que la muestra es
pequeña y pertenece al conjunto de entrenamiento, utilizado durante el desarrollo del
pipeline y del prompt del verificador. Su finalidad es principalmente diagnóstica: comprobar
el comportamiento del pipeline completo sobre ejemplos no inspeccionados previamente e
identificar posibles fuentes de error antes de realizar la evaluación sobre el conjunto
de desarrollo (`dev`).

Los errores observados no presentan un único patrón. En algunos casos, el sistema devuelve
`NOT_ENOUGH_EVIDENCE` junto con `evidence_sufficient=False` para afirmaciones cuyo gold
label es `REFUTED`. Esto puede indicar que el problema no se encuentra necesariamente en
el razonamiento del LLM, sino en una etapa anterior del pipeline: las evidencias recuperadas
por el RAG podrían no contener la información necesaria para refutar la afirmación.

En otros casos, el sistema considera que existe evidencia suficiente, pero asigna una clase
distinta de la etiqueta gold. Estos errores pueden estar relacionados con la interpretación
de las evidencias por parte del `Evidence Verifier`, por ejemplo al distinguir entre
evidencia que contradice directamente una afirmación y evidencia que simplemente describe
una situación alternativa, o al tratar afirmaciones que contienen matices temporales,
cuantitativos o múltiples componentes factuales.

Por este motivo, antes de modificar nuevamente el prompt o los parámetros de recuperación,
se realizará un análisis individual de las predicciones incorrectas. El objetivo será
distinguir entre dos tipos principales de error:

1. **Errores de retrieval:** la evidencia necesaria existe en el Knowledge Store, pero no
   aparece entre los fragmentos recuperados por el sistema.

2. **Errores de verificación:** las evidencias recuperadas contienen información suficiente,
   pero el LLM interpreta incorrectamente su relación con la afirmación o selecciona un
   veredicto diferente al gold.

Esta separación permitirá identificar qué componente del pipeline debe mejorarse sin
modificar innecesariamente etapas que estén funcionando correctamente.

In [57]:
evaluation_results_df["correct"].value_counts()

correct
True     7
False    5
Name: count, dtype: int64

In [59]:
errors_df = evaluation_results_df[
    evaluation_results_df["correct"] == False
][
    [
        "claim_id",
        "claim",
        "gold",
        "predicted",
        "evidence_sufficient",
        "explanation",
    ]
]

# errors_df

In [60]:
for _, row in errors_df.iterrows():

    print("\n" + "=" * 80)

    print("CLAIM ID:", row["claim_id"])
    print("CLAIM:", row["claim"])
    print("GOLD:", row["gold"])
    print("PREDICTED:", row["predicted"])
    print(
        "EVIDENCE SUFFICIENT:",
        row["evidence_sufficient"]
    )

    print("\nEXPLANATION:")
    print(row["explanation"])


CLAIM ID: 929
CLAIM: Government of India is monitoring all forms of communications, both online and telephonic.
GOLD: CONFLICTING_EVIDENCE
PREDICTED: SUPPORTED
EVIDENCE SUFFICIENT: True

EXPLANATION:
The evidence consistently describes India’s Centralized Monitoring System as giving government or law-enforcement agencies centralized, direct capabilities to intercept and monitor telephone communications and Internet activity. Several items explicitly characterize this capability as covering all phone and Internet communications or all telecom networks in India. Although some sources describe the system as being rolled out, in pilot mode, or planned at the time of publication, the evidence sufficiently supports the claim’s broad assertion that the Government of India monitors online and telephonic communications.

CLAIM ID: 777
CLAIM: Billionaire philanthropist and Microsoft founder Bill Gates worked  specifically to end livestock production.
GOLD: NOT_ENOUGH_EVIDENCE
PREDICTED: REFUTED

#### Primera separación diagnóstica según `evidence_sufficient`

Queremos mirar qué dice AVeriTeC en su gold justification para entender qué aspecto semántico ha interpretado de otra forma nuestro verifier.

Aqui 233 y 574 quedan excluídas de este primer bloque porque ambas tienen: `evidence_sufficient=False` y aqui la hipótesis inicial es distinta: quizá el verifier no puede responder porque el RAG no le ha entregado la evidencia adecuada.

In [ ]:
for claim_id in [929, 777, 845]:

    print("\n" + "=" * 80)
    print("CLAIM ID:", claim_id)

    print("\nCLAIM:")
    print(train_df.loc[claim_id, "claim"])

    print("\nGOLD:")
    print(train_df.loc[claim_id, "label"])

    print("\nGOLD JUSTIFICATION:")
    print(train_df.loc[claim_id, "justification"])


CLAIM ID: 929

CLAIM:
Government of India is monitoring all forms of communications, both online and telephonic.

GOLD:
Conflicting Evidence/Cherrypicking

GOLD JUSTIFICATION:
Government of India monitors some forms of communications, but not all of them.

CLAIM ID: 777

CLAIM:
Billionaire philanthropist and Microsoft founder Bill Gates worked  specifically to end livestock production.

GOLD:
Not Enough Evidence

GOLD JUSTIFICATION:
Theres not enough evidence for the claim to be proven or unproven, whilst one answer finds that Bill Gates has back a number of star up meat alternative projects in an attempt to lower carbon footprints theres, no answer could be found on if he wants to end the livestock production altogether so theres not enough evidence presented to proove the claim is true or false.

CLAIM ID: 845

CLAIM:
Being long term unemployed for a year or over nearly halves your chances of ever getting back into employment.

GOLD:
Supported

GOLD JUSTIFICATION:
The first question

Analizamos ahora, otra vez, 845 aparte porque su gold justification es demasiado simple:

Vamos a mirar el primer question/answer gold y analizaremos más adelante los top-10 que recupera nuestro retrieval:

In [63]:
questions_845 = train_df.loc[845, "questions"]

print("Número de preguntas:", len(questions_845))

print("\nFIRST GOLD QUESTION:")
print(questions_845[0]["question"])

print("\nFIRST GOLD ANSWER(S):")

for answer in questions_845[0]["answers"]:
    print("\nAnswer:")
    print(answer["answer"])

    print("\nSource URL:")
    print(answer["source_url"])

Número de preguntas: 2

FIRST GOLD QUESTION:
Does being out of unemployment for a year or over halve your chances of ever getting back into employment?

FIRST GOLD ANSWER(S):

Answer:
Recent analysis suggests people unemployed for one to two years have a 44 per cent smaller chance of finding work within the next year than people unemployed for less than three months. For those unemployed longer, the likelihood is even slimmer.

Source URL:
Metadata


### Análisis de las predicciones incorrectas

La evaluación exploratoria sobre 12 afirmaciones permitió identificar cinco predicciones
incorrectas. Antes de modificar el `Evidence Verifier` o los parámetros del sistema de
recuperación, se analizaron individualmente estos casos con el objetivo de determinar
en qué etapa del pipeline podía haberse originado el error.

Esta distinción es relevante porque una predicción final incorrecta no implica
necesariamente que el LLM haya razonado incorrectamente. El pipeline contiene dos etapas
principales antes de generar el veredicto:

`Knowledge Store → Retrieval → Evidencias recuperadas → Evidence Verifier → Veredicto`

Por tanto, un error puede producirse porque el sistema de recuperación no haya seleccionado
la evidencia necesaria, o porque el `Evidence Verifier` haya interpretado incorrectamente
evidencias que sí eran suficientes.

| Claim ID | Gold | Predicción | Diagnóstico preliminar |
|---|---|---|---|
| 929 | `CONFLICTING_EVIDENCE` | `SUPPORTED` | Posible error del `Evidence Verifier` |
| 777 | `NOT_ENOUGH_EVIDENCE` | `REFUTED` | Posible error del `Evidence Verifier` |
| 233 | `REFUTED` | `NOT_ENOUGH_EVIDENCE` | Posible error de retrieval |
| 574 | `REFUTED` | `NOT_ENOUGH_EVIDENCE` | Posible error de retrieval |
| 845 | `SUPPORTED` | `CONFLICTING_EVIDENCE` | Error todavía ambiguo entre retrieval y verificación |

#### Claim 929

La afirmación sostiene que el Gobierno de India monitoriza **todas** las formas de
comunicación, tanto online como telefónicas. El gold label de AVeriTeC es
`CONFLICTING_EVIDENCE` y su justificación indica que el Gobierno monitoriza algunas formas
de comunicación, pero no todas.

El `Evidence Verifier`, sin embargo, clasificó la afirmación como `SUPPORTED`. En su propia
explicación reconoció que algunas fuentes describían el sistema como planificado, en fase
piloto o todavía en proceso de despliegue, pero terminó considerando suficiente la evidencia
para respaldar la afirmación completa.

Este caso sugiere un posible problema de interpretación relacionado con el alcance de la
afirmación. Expresiones como *some*, *certain* o la existencia de capacidad para realizar
una actividad no implican necesariamente una afirmación absoluta como *all forms of
communications*. Por tanto, el verificador debe prestar especial atención a cuantificadores,
alcance y estado temporal de los hechos.

#### Claim 777

La afirmación indica que Bill Gates trabajó específicamente para terminar con la producción
ganadera. AVeriTeC la clasifica como `NOT_ENOUGH_EVIDENCE`, ya que existen evidencias sobre
su apoyo a proyectos de alternativas a la carne y reducción de emisiones, pero no evidencia
suficiente para determinar si su objetivo era eliminar completamente la producción ganadera.

El sistema predijo `REFUTED` porque recuperó evidencias sobre financiación de sanidad animal,
productividad ganadera y medios de vida relacionados con el ganado, interpretando estas
actividades como una contradicción directa de la afirmación.

Sin embargo, la existencia de actividades diferentes o aparentemente opuestas no constituye
por sí sola una refutación. Para clasificar una afirmación como `REFUTED`, la evidencia debe
contradecir directamente el hecho afirmado o hacerlo lógicamente incompatible. Este caso
sugiere que el `Evidence Verifier` puede estar utilizando un criterio demasiado amplio para
considerar que una evidencia refuta una afirmación.

#### Claims 233 y 574

Las claims 233 y 574 presentan un patrón diferente. Ambas tienen gold label `REFUTED`, pero
el sistema devuelve `NOT_ENOUGH_EVIDENCE` junto con `evidence_sufficient=False`.

En la claim 233, relativa a cifras de policías heridos, fallecidos y daños económicos
atribuidos a BLM, el verificador indica que las evidencias recuperadas mencionan incidentes
individuales, pero no proporcionan información suficiente para confirmar ni refutar las
cantidades concretas indicadas en la afirmación.

En la claim 574, relativa a un supuesto tweet de Walmart, el verificador señala que ninguna
de las evidencias recuperadas permite verificar que el mensaje existiera ni demuestra
directamente que fuera falso.

En ambos casos, la conclusión `NOT_ENOUGH_EVIDENCE` es coherente con las evidencias que el
LLM afirma haber recibido. Por ello, estos errores podrían encontrarse en una etapa anterior:
la evidencia necesaria para refutar las afirmaciones puede existir dentro del Knowledge
Store de AVeriTeC, pero no haber sido seleccionada entre los fragmentos finales recuperados
por el sistema. Estos casos se analizarán posteriormente comparando las evidencias
recuperadas con las evidencias de referencia del dataset.

#### Claim 845

La claim 845 afirma que permanecer desempleado durante un año o más reduce casi a la mitad
la probabilidad de volver a encontrar empleo. El sistema la clasificó como
`CONFLICTING_EVIDENCE`, mientras que AVeriTeC la etiqueta como `SUPPORTED`.

La justificación general del dataset únicamente indicaba que el primer par
pregunta-respuesta respaldaba la afirmación, por lo que se inspeccionó específicamente dicha
evidencia gold. La respuesta de referencia indica que las personas desempleadas durante
entre uno y dos años tienen una probabilidad un **44 % menor** de encontrar trabajo durante
el año siguiente que aquellas desempleadas durante menos de tres meses, y que la probabilidad
disminuye todavía más para periodos de desempleo superiores.

Esta evidencia respalda claramente los elementos centrales de la claim:

`one year or over → unemployed for one to two years`

y

`nearly halves the chances → 44 % smaller chance`.

Existe una pequeña diferencia entre la formulación *ever getting back into employment* y la
evidencia, que se refiere a encontrar empleo durante el año siguiente. Sin embargo, los
anotadores de AVeriTeC consideraron que la correspondencia semántica era suficiente para
clasificar la afirmación como `SUPPORTED`.

El `Evidence Verifier` había considerado conflictiva la evidencia porque también encontró
información indicando que una parte importante de los desempleados de larga duración vuelve
a encontrar trabajo. No obstante, estas dos afirmaciones no son necesariamente
contradictorias: es posible que muchas personas vuelvan a trabajar y, simultáneamente, que
su probabilidad de hacerlo sea considerablemente inferior a la de personas que llevan menos
tiempo desempleadas.

Este caso muestra que `CONFLICTING_EVIDENCE` no debe utilizarse simplemente porque existan
estadísticas o formulaciones diferentes. Debe existir una contradicción factual relevante
entre evidencias que afecte al veredicto de la claim.

No obstante, todavía debe comprobarse si la evidencia gold del 44 % fue realmente recuperada
entre los fragmentos enviados al LLM. Si estaba presente, el error correspondería
principalmente al `Evidence Verifier`; si no estaba presente, el sistema de retrieval habría
contribuido al error.

En consecuencia, esta primera evaluación muestra que los fallos del sistema no proceden
necesariamente de un único componente. Algunos casos apuntan a problemas de recuperación de
evidencia, mientras que otros revelan posibles limitaciones en la interpretación semántica
del `Evidence Verifier`. Por este motivo, no se modifica todavía el prompt ni la estrategia
de recuperación: primero se completará el diagnóstico de los errores para evitar corregir
un componente cuando el problema se encuentre en otro.

Seguimos verificando la claim 845, 233 y 574:

In [64]:
claim_id = 845
claim = train_df.loc[claim_id, "claim"]

result_845 = verify_averitec_claim(
    claim_id=claim_id,
    claim=claim,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus manejable: 38173 chunks.
No es necesario aplicar preselección TF-IDF.
Documentos originales: 623
Documentos únicos: 623
Documentos finales: 623
Chunks iniciales: 38173
Chunks finales: 38173


Batches: 100%|██████████| 597/597 [1:43:37<00:00, 10.41s/it]


In [65]:
for i, evidence in enumerate(
    result_845["evidences"],
    start=1
):
    print("\n" + "=" * 80)
    print(f"EVIDENCE {i}")
    print("Score:", round(evidence["score"], 4))
    print("URL:", evidence["url"])
    print("\nTEXT:")
    print(evidence["text"])


EVIDENCE 1
Score: 0.805
URL: https://time.com/34114/unemployment-hiring-rate/

TEXT:
Only one in 10 long-term people who have been unemployed for six months or more are hired every year, according to a Princeton study that affirms the bleak outlook for many long-term job seekers. With the unemployment rate near five-year lows amid a post-recession recovery, the long-term unemployed are still struggling to find jobs. Even in regions with the strongest job markets, the rate of hiring for the long-term jobless is no better. Alan Krueger, lead researcher and a former chief White House economic advisor, told CNBC that long-term job seekers are prone to becoming discouraged and putting less effort into finding a job. Meanwhile, employers can be skeptical of potential hires who have not worked for six months. “A concerted effort will be needed to raise the employment prospects of the long-term unemployed, especially as they are likely to withdraw from the job market at an increasing rate,” K

La conclusión importante es: no es un error puramente de retrieval ni puramente del verifier; hay contribución de ambos.

La evidencia que Averitec utilizó decía: 1–2 años desempleado, implica 44 % menos probabilidad de encontrar trabajo. Esa formulación concreta no aparece en nuestro Top-10. Nuestro RAG recuperó documentos `muy relacionados`, pero no ese fragmento exacto.

Un problema que vemos: El verifier utilizó especialmente Evidence 5 para justificar `CONFLICTING_EVIDENCE`. El verifier está tomando estadísticas diferentes como contradictorias. 

`Conclusión:` El Retrieval recupera información muy relevante, pero no el fragmento gold específico del 44 %. Por otro lado el Evidence Verifier interpreta como contradictorias evidencias que pueden coexistir perfectamente.

In [66]:
claim_id = 233
claim = train_df.loc[claim_id, "claim"]

result_233 = verify_averitec_claim(
    claim_id=claim_id,
    claim=claim,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 150586 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 1115
Documentos únicos: 1112
Documentos finales: 100
Chunks iniciales: 150586
Chunks finales: 8933


Batches: 100%|██████████| 140/140 [41:39<00:00, 17.86s/it]


In [67]:
for i, evidence in enumerate(
    result_233["evidences"],
    start=1
):
    print("\n" + "=" * 80)
    print(f"EVIDENCE {i}")
    print("Score:", round(evidence["score"], 4))
    print("URL:", evidence["url"])
    print("\nTEXT:")
    print(evidence["text"])


EVIDENCE 1
Score: 0.65
URL: https://www.pewresearch.org/social-trends/2017/01/11/behind-the-badge/

TEXT:
police say their jobs are harder now as a consequence of recent high-profile fatal incidents involving blacks and police. Overall, fully 86% of officers say their jobs are harder, including substantial majorities of officers in police departments with fewer than 300 officers as well as those working in “mega departments” with 2,600 officers or more (84% and 89%, respectively). In fact, across every major demographic group analyzed for this survey, about eight-in-ten officers or more say these high-profile incidents have made policing more challenging and more dangerous. While the impact of these incidents is broadly felt, officers in larger departments are far more likely than those in small agencies to say these incidents have had an impact. For example, roughly half of officers (54%) in departments with fewer than 300 officers say their peers have become less willing to stop and

Viendo el Top-10 de 233, la respuesta del LLM tiene bastante sentido. Para poder marcarla como REFUTED, idealmente necesitaríamos evidencia que contradiga una o varias de las cifras de la claim de manera suficientemente directa, sin embargo, nuestro Top-10 contiene principalmente información relacionada con protestas, policía y disturbios, pero no las cifras concretas de la claim. 

Aqui diferente de la claim 845 el retrieval no ha recuperado información suficiente, por tanto aqui el problema se encuentra en la etapa de recuperación de evidencia, cuya causa concreta
se analizará posteriormente. Otro detalle importante aqui son los scores recuperados, que están alrededor del 0.6, mientras que en la claim 845 estaban alrededor de 0.8; Esto no implica que debemos convertir estas evidencias en malas porque el score no tiene un umbral universal de verdad o calidad, pero comparativamente, sí es coherente con lo que estamos viendo: el retriever parece haber encontrado coincidencias semánticas mucho menos alineadas con la claim 233 que con la 845.

Para la claim 233 todavía nos falta una verificación, ¿Qué evidencia utilizó AVeriTeC para determinar que la claim era Refuted?

In [70]:
claim_id = 233

print("CLAIM:")
print(train_df.loc[claim_id, "claim"])

print("\nGOLD:")
print(train_df.loc[claim_id, "label"])

print("\nGOLD JUSTIFICATION:")
print(train_df.loc[claim_id, "justification"])

print("\nQUESTIONS AND ANSWERS:")

questions = train_df.loc[claim_id, "questions"]

for i, qa in enumerate(questions, start=1):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}:")
    print(qa["question"])

    print("\nANSWERS:")

    for answer in qa["answers"]:
        print("\nAnswer:")
        print(answer["answer"])

        print("\nSource:")
        print(answer["source_url"])

CLAIM:
BLM injures 1000 police officers, kills36 people and does $8 billion in damage .

GOLD:
Refuted

GOLD JUSTIFICATION:
Only about 19 people have been confirmed dead as a result of the protests and about 700 police officers nationwide 

QUESTIONS AND ANSWERS:

QUESTION 1:
Have there been news reports about how many people have been injured?

ANSWERS:

Answer:
WASHINGTON — More than 700 law enforcement officers have been injured on the job during nationwide protests over the death of George Floyd — with nearly 300 of those among New York’s Finest, according to the Department of Justice and the NYPD.

Source:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/

QUESTION 2:
Is there any estimates of how much it has cost in damages?

ANSWERS:

Answer:
Not an exact amount but estimates are made.

Prior to 2020, the costliest civil disorder event in U.S. was the 1992 Los Angeles riots, according to PCS, the insurance industry's primary source 

La diferencia más importante aqui es que nuestro Evidence Verifier no recibió estas evidencias concretas, y respecto a los daños, nuestro retrieval sí recuperó información relacionada. 

La cadena de errores sería: Knowledge Store de AVeriTeC, contiene evidencia útil: 700+ officers / 19 deaths, pero nuestro retrieval, NO selecciona esos fragmentos en el Top-10. El Evidence Verifier recibe información relacionada pero no decisiva
y determina `NOT_ENOUGH_EVIDENCE` con `evidence_sufficient=False`. Por lo tanto, en esta claim concluimos que el problema es del retrieval y no del verifier. 

En este caso, no deberíamos arreglar este problema modificando el prompt para que el LLM “adivine” que 1.000 o 36 son falsas. El problema está antes: necesitamos aumentar la probabilidad de recuperar evidencia que responda específicamente a las distintas partes de una claim numérica y multicomponente.

In [68]:
claim_id = 574
claim = train_df.loc[claim_id, "claim"]

result_574 = verify_averitec_claim(
    claim_id=claim_id,
    claim=claim,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 63998 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 850
Documentos únicos: 850
Documentos finales: 100
Chunks iniciales: 63998
Chunks finales: 2243


Batches: 100%|██████████| 36/36 [08:18<00:00, 13.85s/it]


In [69]:
for i, evidence in enumerate(
    result_574["evidences"],
    start=1
):
    print("\n" + "=" * 80)
    print(f"EVIDENCE {i}")
    print("Score:", round(evidence["score"], 4))
    print("URL:", evidence["url"])
    print("\nTEXT:")
    print(evidence["text"])


EVIDENCE 1
Score: 0.7229
URL: https://www.ufcw.org/press-releases/protests-for-better-jobs-at-walmart-sweep-stores-nationwide/

TEXT:
said Dorothy Halvorson, a Walmart employee in Placerville, California, who has worked at the store for 11 years and plans to take part in civil disobedience today. “We know that we are closer to change at Walmart than ever before – and it’s clear that Walmart knows it too. We won’t stop protesting until we get change. This Black Friday is historic, and we will only grow stronger from here.” In recent weeks, protesting Walmart workers have received an outpouring of national support. Calling for an end to Walmart’s illegal retaliation and attempts to silence workers who speak out for better working conditions, workers have walked off their jobs in a dozen cities, including Los Angeles, Seattle, Chicago, Ohio, Dallas, Florida, Washington, D.C. and elsewhere. “The civil disobedience by Walmart workers and supporters across the country is a testament to the 

Esta claim 574 parece todavía más clara que la 233: con el Top-10 que recuperó nuestro RAG, el Evidence Verifier tenía muy difícil llegar a `REFUTED`.

Al mirar las 10 evidencias, prácticamente todas hablan de la relación de Walmart con la policía, seguridad, colaboración con fuerzas del orden, amenazas o llamadas policiales en tiendas, pero ninguna habla del supuesto tweet ni de si fue verdadero o falso. 

El retriever encontró contenido semánticamente relacionado con los términos de la claim, pero no evidencia sobre la autenticidad de la publicación, por eso el LLM concluyó antes algo como: No tengo ninguna evidencia que reproduzca,
atribuya o desmienta el supuesto tweet: `NOT_ENOUGH_EVIDENCE`. Así que, igual que con la 233, el problema apunta muy claramente al retrieval.

¿Qué evidencia utilizó AVeriTeC para determinar que la claim era Refuted?

In [71]:
claim_id = 574

print("CLAIM:")
print(train_df.loc[claim_id, "claim"])

print("\nGOLD:")
print(train_df.loc[claim_id, "label"])

print("\nGOLD JUSTIFICATION:")
print(train_df.loc[claim_id, "justification"])

print("\nQUESTIONS AND ANSWERS:")

questions = train_df.loc[claim_id, "questions"]

for i, qa in enumerate(questions, start=1):

    print("\n" + "=" * 80)
    print(f"QUESTION {i}:")
    print(qa["question"])

    print("\nANSWERS:")

    for answer in qa["answers"]:
        print("\nAnswer:")
        print(answer["answer"])

        print("\nSource:")
        print(answer["source_url"])

CLAIM:
Walmart Tweeted "We wont be Rebuilding Walmarts in Cities without police"

GOLD:
Refuted

GOLD JUSTIFICATION:
The claim has been shown to be fake and no results from Walmart show that statement. 

QUESTIONS AND ANSWERS:

QUESTION 1:
Did Walmart tweet "We wont be Rebuilding Walmarts in Cities without police"

ANSWERS:

Answer:
No

Source:
https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query

QUESTION 2:
What tweet does the claim refer to?

ANSWERS:

Answer:
A facebook post from Facebook user M. Cody Clifton

Source:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater

QUESTION 3:
Is the post from Facebook user M. Cody Clifton confirmed real?

ANSWERS:

Answer:
False, the tweet appears to be fabricated

Source:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&s

AVeriTeC dispone de evidencia que ataca exactamente el núcleo de la claim, es decir el tweet no aparece en Walmart, la imagen/post procede de otro usuario y el contenido está fabricado, esto justifica `REFUTED`. En cambio, nuestro Top-10 no contenía nada de eso. Recuperó páginas sobre Walmart y policía, seguridad, llamadas policiales, amenazas, agentes fuera de servicio, etc, por eso el verifier dijo: `NOT_ENOUGH_EVIDENCE`, evidence_sufficient = False.

La cadena de error queda clara, Knowledge Store contiene evidencia que demuestra: "tweet fabricated" pero retrieve_evidence NO selecciona esos fragmentos, por lo tanto Evidence Verifier solo recibe contenido relacionado
con Walmart + policía y no puede refutar el tweet y concluir `NOT_ENOUGH_EVIDENCE`. Por lo tanto, en esta fase se concluye que el error se origina antes del `Evidence Verifier`, dentro del proceso de recuperación/preparación de evidencia.
La causa concreta se analiza posteriormente en el diagnóstico final de retrieval.

Además, 574 refuerza algo que ya vimos en 233: nuestro retrieval funciona bien para encontrar contenido temáticamente relacionado, pero no siempre encuentra la evidencia que responde exactamente a la pregunta verificadora.

### 13. Modificaciones en el sistema Global

El análisis de las predicciones incorrectas permitió identificar tres patrones generales
relacionados con la interpretación de las evidencias por parte del `Evidence Verifier`:
el tratamiento insuficiente de cuantificadores y alcance, la consideración de actividades
alternativas como refutaciones directas y el uso de `CONFLICTING_EVIDENCE` ante evidencias
diferentes pero no necesariamente contradictorias.

A partir de estos resultados se introduce una tercera versión del prompt (`V3`) que refuerza
estos criterios sin modificar las clases de salida, la estructura Pydantic ni el pipeline
de retrieval. Los cambios se formulan como reglas generales y no como correcciones
específicas para las claims analizadas.

El objetivo no es optimizar el prompt para acertar los ejemplos de entrenamiento, sino
corregir patrones de razonamiento generalizables detectados durante el análisis de errores.

In [ ]:
def verify_evidence(
    claim,
    evidences,
    client,
    model="gpt-5.6-terra",
    language=None,
):
    """
    Verifica una claim utilizando exclusivamente las evidencias recuperadas por el sistema RAG.

    Parameters
    ----------
    claim : str
        Afirmación que se quiere verificar.

    evidences : list[dict]
        Evidencias recuperadas por retrieve_evidence().

    client
        Cliente de OpenAI.

    model : str
        Modelo LLM utilizado para realizar la verificación.

    Returns
    -------
    VerificationResult
        Resultado estructurado de la verificación factual.
    """

    # Preparamos las evidencias recuperadas en un formato comprensible para el LLM
    evidence_text = format_evidences_for_llm(evidences)

    if language == "es":
        language_instruction = (
            "Write the explanation and reasons in Spanish."
        )
    elif language == "en":
        language_instruction = (
            "Write the explanation and reasons in English."
        )

    instructions = f"""
    You are an Evidence Verifier for a fact-checking system.

    Your task is to verify a claim using ONLY the evidence provided to you.

    Do not use external knowledge.
    Do not assume facts that are not present in the evidence.

    A claim may contain multiple factual components.
    When this happens, consider the different components of the claim before assigning evidence relationships and the 
    final verdict.

    Pay careful attention to quantifiers, scope, and temporal status.
    Evidence supporting "some", "many", "certain", "planned", "pilot", "partially deployed", or similar limited statements
    does not automatically support a claim asserting "all", "always", "fully implemented", or another broader or absolute
    statement.

    For each evidence item, determine its relationship to the claim:

    - SUPPORTS:
    The evidence provides factual information that supports the claim as a whole, or supports an important component without 
    contradicting or leaving unresolved another essential component of the claim.

    - REFUTES:
    The evidence provides factual information that contradicts the claim as a whole, or directly contradicts an essential
    component in a way that makes the overall claim false.

    Do not classify evidence as REFUTES merely because it describes a different, alternative, or apparently opposing  
    activity. REFUTES requires a direct contradiction or a factual relationship that is logically incompatible
    with the claim.

    - NEUTRAL:
    The evidence is related to the claim but does not provide enough information to support or refute it as a whole.

    Use NEUTRAL when the evidence addresses only one component of a multi-part claim and leaves other essential components
    unresolved.

    Then determine the overall verdict:

    - SUPPORTED:
    The available evidence is sufficient to support the claim as a whole.

    - REFUTED:
    The available evidence is sufficient to refute the claim as a whole.

    - NOT_ENOUGH_EVIDENCE:
    The available evidence does not contain enough information to reach a justified conclusion about the claim.

    - CONFLICTING_EVIDENCE:
    The available evidence supports some important factual aspects of the claim while refuting, contradicting, or 
    substantially qualifying other important aspects, so that labeling the whole claim simply SUPPORTED or REFUTED would
    lose relevant information.

    Use this verdict also for cherry-picking cases, where the claim relies on factually supported information but 
    selectively combines, omits, or frames evidence in a way that produces a misleading overall conclusion.

    Do not use CONFLICTING_EVIDENCE merely because different evidence items contain different statistics, perspectives, 
    wording, or levels of detail.
    The evidence must contain a meaningful factual incompatibility that affects the truth or interpretation of the claim.

    Do not automatically label a multi-part claim REFUTED simply because one component is contradicted if other important
    components are supported.
    In such cases, consider whether CONFLICTING_EVIDENCE is more appropriate.

    Set evidence_sufficient to true when the provided evidence contains enough factual information to justify the selected 
    verdict, including when the evidence is sufficient to establish that the case is CONFLICTING_EVIDENCE.

    Set evidence_sufficient to false when the appropriate verdict is NOT_ENOUGH_EVIDENCE.

    Assess every evidence item provided.

    Base the final verdict on the content of the evidence, not on retrieval similarity scores.

    {language_instruction}
    """
    response = client.responses.parse(
        model=model,
        instructions=instructions,
        input=f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {evidence_text}
    """,
            text_format=VerificationResult,
        )

    return response.output_parsed

Vamos a probar estas claims. Las claims 929,777 y 845 para ver si corrige los tres patrones que motivaron el cambio, y 275
770, 349 y 611 porque queremos comprobar que prompt V3 NO rompe casos que prompt V2 ya resolvía correctamente

In [72]:
regression_claim_ids = [
    929,  # Error V2: CONFLICTING -> SUPPORTED
    777,  # Error V2: NEI -> REFUTED
    845,  # Error V2: SUPPORTED -> CONFLICTING

    275,  # CONFLICTING correcta previamente
    770,  # NEI correcta previamente
    349,  # REFUTED correcta previamente
    611,  # SUPPORTED correcta previamente
]

In [73]:
regression_sample = train_df.loc[
    regression_claim_ids
].copy()

El objetivo de este regression check no es obtener una nueva estimación de rendimiento,
sino comprobar si las modificaciones del prompt corrigen los patrones detectados sin
degradar ejemplos que la versión anterior clasificaba correctamente.

In [79]:
regression_results_df = evaluate_averitec_sample(
    evaluation_sample=regression_sample,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
    output_path=PROCESSED_DIR / "regression_v3_results.csv",
)

Resultados anteriores encontrados: 5 claims.

Claim 929 ya evaluada. Se omite.

Claim 777 ya evaluada. Se omite.

Claim 845 ya evaluada. Se omite.

Claim 275 ya evaluada. Se omite.

Claim 770 ya evaluada. Se omite.

Evaluando claim 349...
Gold: REFUTED
Corpus grande detectado: 43075 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 589
Documentos únicos: 589
Documentos finales: 100
Chunks iniciales: 43075
Chunks finales: 7702


Batches: 100%|██████████| 121/121 [21:02<00:00, 10.44s/it]


Predicted: REFUTED
Correct: True
Resultados guardados: 6

Evaluando claim 611...
Gold: SUPPORTED
Corpus grande detectado: 81000 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 733
Documentos únicos: 733
Documentos finales: 100
Chunks iniciales: 81000
Chunks finales: 3063


Batches: 100%|██████████| 48/48 [06:21<00:00,  7.95s/it]


Predicted: SUPPORTED
Correct: True
Resultados guardados: 7


In [80]:
regression_results_df[
    [
        "claim_id",
        "gold",
        "predicted",
        "correct",
        "evidence_sufficient",
    ]
]

,claim_id,gold,predicted,correct,evidence_sufficient
0,929,CONFLICTING_EVIDENCE,CONFLICTING_EVIDENCE,True,True
1,777,NOT_ENOUGH_EVIDENCE,REFUTED,False,True
2,845,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,False
3,275,CONFLICTING_EVIDENCE,SUPPORTED,False,True
4,770,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,False
5,349,REFUTED,REFUTED,True,True
6,611,SUPPORTED,SUPPORTED,True,True


In [81]:
regression_results_df["correct"].value_counts()

correct
True     4
False    3
Name: count, dtype: int64

Vamos a leer las explicaciones de las 3 claims que todavía siguen mal:

Con esto podemos ver si V3 necesita algun ajuste o si alguna de las nuevas reglas está creando más problemas de los que resuelve.

In [82]:
for claim_id in [777, 845, 275]:

    row = regression_results_df[
        regression_results_df["claim_id"] == claim_id
    ].iloc[0]

    print("\n" + "=" * 80)
    print("CLAIM ID:", claim_id)

    print("\nCLAIM:")
    print(row["claim"])

    print("\nGOLD:")
    print(row["gold"])

    print("\nPREDICTED:")
    print(row["predicted"])

    print("\nEVIDENCE SUFFICIENT:")
    print(row["evidence_sufficient"])

    print("\nEXPLANATION:")
    print(row["explanation"])


CLAIM ID: 777

CLAIM:
Billionaire philanthropist and Microsoft founder Bill Gates worked  specifically to end livestock production.

GOLD:
NOT_ENOUGH_EVIDENCE

PREDICTED:
REFUTED

EVIDENCE SUFFICIENT:
True

EXPLANATION:
The evidence consistently shows Gates and the Gates Foundation funding, promoting, or investigating measures to improve livestock breeding, health, productivity, and veterinary access, especially for poor farmers. These activities are directly incompatible with the claim that he worked specifically to end livestock production.

CLAIM ID: 845

CLAIM:
Being long term unemployed for a year or over nearly halves your chances of ever getting back into employment.

GOLD:
SUPPORTED

PREDICTED:
NOT_ENOUGH_EVIDENCE

EVIDENCE SUFFICIENT:
False

EXPLANATION:
The evidence consistently indicates that prolonged unemployment is associated with worse hiring or interview prospects. However, it does not provide a like-for-like comparison showing that unemployment lasting a year or more 

#### Refinamiento del prompt tras el regression check

El regression check mostró que la versión V3 corregía algunos errores de interpretación,
pero también introducía nuevos problemas. En particular, el verifier seguía considerando
como refutación algunas actividades alternativas que no constituían una contradicción
directa, exigía en algunos casos una correspondencia demasiado literal entre claim y
evidencia, y podía dejar de detectar conflictos cuando una limitación modificaba de forma
significativa la interpretación de la afirmación.

Por este motivo se realiza un ajuste menor del prompt (`V3.1`). La nueva versión refuerza
que una refutación debe basarse en una contradicción factual explícita, que permite considerar
evidencia semánticamente equivalente aunque no reproduzca literalmente la claim y aclara
que `CONFLICTING_EVIDENCE` puede utilizarse cuando una evidencia introduce una limitación
o matización que altera de forma material el significado de la afirmación.

No se modifican las clases de salida, la estructura Pydantic ni el componente de retrieval.

In [ ]:
def verify_evidence(
    claim,
    evidences,
    client,
    model="gpt-5.6-terra",
    language=None,
):
    """
    Verifica una claim utilizando exclusivamente las evidencias recuperadas por el sistema RAG.

    Parameters
    ----------
    claim : str
        Afirmación que se quiere verificar.

    evidences : list[dict]
        Evidencias recuperadas por retrieve_evidence().

    client
        Cliente de OpenAI.

    model : str
        Modelo LLM utilizado para realizar la verificación.

    Returns
    -------
    VerificationResult
        Resultado estructurado de la verificación factual.
    """

    # Preparamos las evidencias recuperadas en un formato comprensible para el LLM
    evidence_text = format_evidences_for_llm(evidences)

    if language == "es":
        language_instruction = (
            "Write the explanation and reasons in Spanish."
        )
    elif language == "en":
        language_instruction = (
            "Write the explanation and reasons in English."
        )
  
    instructions = f"""
    You are an Evidence Verifier for a fact-checking system.

    Your task is to verify a claim using ONLY the evidence provided to you.

    Do not use external knowledge.
    Do not assume facts that are not present in the evidence.

    A claim may contain multiple factual components.
    When this happens, consider the different components of the claim before assigning evidence relationships and the 
    final verdict.

    Pay careful attention to quantifiers, scope, and temporal status.
    Evidence supporting "some", "many", "certain", "planned", "pilot", "partially deployed", or similar limited statements
    does not automatically support a claim asserting "all", "always", "fully implemented", or another broader or absolute 
    statement.

    For each evidence item, determine its relationship to the claim:

    - SUPPORTS:
    The evidence provides factual information that supports the claim as a whole, or supports an important component without
    contradicting or leaving unresolved another essential component of the claim.

    Evidence does not need to reproduce the exact wording of the claim.
    It may SUPPORT the claim when it expresses the same relevant factual meaning using different wording, measurements, 
    time windows, or formulations, provided that those differences do not materially change the factual conclusion.

    - REFUTES:
    The evidence provides factual information that directly contradicts the claim as a whole, or directly contradicts an
    essential component in a way that makes the overall claim false.

    Do not classify evidence as REFUTES merely because it describes a different, alternative, opposite-looking, or 
    competing activity.

    REFUTES requires an explicit factual contradiction with the claim or with an essential factual component of it. 
    Do not infer that a claim is false only because the evidence describes behavior that appears inconsistent with
    it unless the evidence itself establishes the contradiction.

    - NEUTRAL:
    The evidence is related to the claim but does not provide enough information to support or refute it as a whole.

    Use NEUTRAL when the evidence addresses only one component of a multi-part claim and leaves other essential components
    unresolved.

    Then determine the overall verdict:

    - SUPPORTED:
    The available evidence is sufficient to support the claim as a whole.

    Do not require an exact lexical or numerical match when the evidence conveys substantially the same factual conclusion
    and the differences do not materially alter the meaning of the claim.

    - REFUTED:
    The available evidence is sufficient to directly contradict the claim as a whole or an essential factual component 
    that makes the overall claim false.

    - NOT_ENOUGH_EVIDENCE:
    The available evidence does not contain enough information to reach a justified conclusion about the claim.

    - CONFLICTING_EVIDENCE:
    The available evidence supports some important factual aspects of the claim while refuting, contradicting, or 
    substantially qualifying other important aspects, so that labeling the whole claim simply SUPPORTED or REFUTED would
    lose relevant information.

    Use this verdict also for cherry-picking cases, where the claim relies on factually supported information but 
    selectively combines, omits, or frames evidence in a way that produces a misleading overall conclusion.

    Do not use CONFLICTING_EVIDENCE merely because different evidence items contain different statistics, perspectives,
    wording, or levels of detail.

    However, a limitation, qualification, difference in scope, or contextual condition should be considered relevant when
    it materially changes the interpretation, strength, or overall meaning of the claim. In such cases, 
    CONFLICTING_EVIDENCE may be more appropriate than SUPPORTED


    Do not automatically label a multi-part claim REFUTED simply because one component is contradicted if other important 
    components are supported. In such cases, consider whether CONFLICTING_EVIDENCE is more appropriate.

    Set evidence_sufficient to true when the provided evidence contains enough factual information to justify the selected
    verdict, including when the evidence is sufficient to establish that the case is CONFLICTING_EVIDENCE.

    Set evidence_sufficient to false when the appropriate verdict is NOT_ENOUGH_EVIDENCE.

    Assess every evidence item provided.

    Base the final verdict on the content of the evidence, not on retrieval similarity scores.

    {language_instruction}
    """
    response = client.responses.parse(
        model=model,
        instructions=instructions,
        input=f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {evidence_text}
    """,
            text_format=VerificationResult,
        )

    return response.output_parsed

In [84]:
regression_results_df = evaluate_averitec_sample(
    evaluation_sample=regression_sample,
    knowledge_zip=knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
    output_path=PROCESSED_DIR / "regression_v3_1_results.csv",
)


Evaluando claim 929...
Gold: CONFLICTING_EVIDENCE
Corpus grande detectado: 46749 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 514
Documentos únicos: 513
Documentos finales: 100
Chunks iniciales: 46749
Chunks finales: 5563


Batches: 100%|██████████| 87/87 [17:45<00:00, 12.25s/it]


Predicted: CONFLICTING_EVIDENCE
Correct: True
Resultados guardados: 1

Evaluando claim 777...
Gold: NOT_ENOUGH_EVIDENCE
Corpus manejable: 36072 chunks.
No es necesario aplicar preselección TF-IDF.
Documentos originales: 629
Documentos únicos: 629
Documentos finales: 629
Chunks iniciales: 36072
Chunks finales: 36072


Batches: 100%|██████████| 564/564 [1:51:39<00:00, 11.88s/it]  


Predicted: REFUTED
Correct: False
Resultados guardados: 2

Evaluando claim 845...
Gold: SUPPORTED
Corpus manejable: 38173 chunks.
No es necesario aplicar preselección TF-IDF.
Documentos originales: 623
Documentos únicos: 623
Documentos finales: 623
Chunks iniciales: 38173
Chunks finales: 38173


...

Predicted: NOT_ENOUGH_EVIDENCE
Correct: True
Resultados guardados: 5

Evaluando claim 349...
Gold: REFUTED
Corpus grande detectado: 43075 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 589
Documentos únicos: 589
Documentos finales: 100
Chunks iniciales: 43075
Chunks finales: 7702


Batches: 100%|██████████| 121/121 [16:19<00:00,  8.10s/it]


Predicted: REFUTED
Correct: True
Resultados guardados: 6

Evaluando claim 611...
Gold: SUPPORTED
Corpus grande detectado: 81000 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 733
Documentos únicos: 733
Documentos finales: 100
Chunks iniciales: 81000
Chunks finales: 3063


Batches: 100%|██████████| 48/48 [07:02<00:00,  8.79s/it]


Predicted: SUPPORTED
Correct: True
Resultados guardados: 7


In [85]:
regression_results_df[
    [
        "claim_id",
        "gold",
        "predicted",
        "correct",
        "evidence_sufficient",
    ]
]

,claim_id,gold,predicted,correct,evidence_sufficient
0,929,CONFLICTING_EVIDENCE,CONFLICTING_EVIDENCE,True,True
1,777,NOT_ENOUGH_EVIDENCE,REFUTED,False,True
2,845,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,False
3,275,CONFLICTING_EVIDENCE,SUPPORTED,False,True
4,770,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,True,False
5,349,REFUTED,REFUTED,True,True
6,611,SUPPORTED,SUPPORTED,True,True


#### Selección de la versión final del Evidence Verifier

El regression check de la versión V3.1 no mostró mejoras adicionales respecto a V3.
Las siete claims utilizadas obtuvieron exactamente las mismas predicciones en ambas
versiones.

Aunque V3 y V3.1 corrigieron la clasificación de la claim 929, no corrigieron los errores
de las claims 777 y 845 y, además, introdujeron una regresión en la claim 275, que había sido
clasificada correctamente por V2.

Por tanto, las modificaciones posteriores a V2 no muestran una mejora global y consistente
del comportamiento del verifier. Para evitar continuar ajustando el prompt sobre un número
reducido de ejemplos del conjunto de entrenamiento, se decide detener el proceso de
optimización y conservar V2 como versión final del `Evidence Verifier`.

Los errores restantes se mantienen como limitaciones observadas del sistema y serán
considerados en el análisis de resultados, sin realizar nuevos ajustes antes de la evaluación
sobre el conjunto `dev`.

#### Configuración final del Evidence Verifier

Tras comparar V2, V3 y V3.1 mediante el regression check, se conserva V2 como versión
final del prompt. A partir de este punto, esta configuración se considera congelada y será
la utilizada durante la evaluación sobre el conjunto `dev`.

In [ ]:
def verify_evidence(
    claim,
    evidences,
    client,
    model="gpt-5.6-terra",
    language=None,
):
    """
    Verifica una claim utilizando exclusivamente las evidencias recuperadas por el sistema RAG.

    Parameters
    ----------
    claim : str
        Afirmación que se quiere verificar.

    evidences : list[dict]
        Evidencias recuperadas por retrieve_evidence().

    client
        Cliente de OpenAI.

    model : str
        Modelo LLM utilizado para realizar la verificación.

    Returns
    -------
    VerificationResult
        Resultado estructurado de la verificación factual.
    """

    # Preparamos las evidencias recuperadas en un formato comprensible para el LLM
    evidence_text = format_evidences_for_llm(evidences)

    if language == "es":
        language_instruction = (
            "Write the explanation and reasons in Spanish."
        )
    elif language == "en":
        language_instruction = (
            "Write the explanation and reasons in English."
        )
   
    instructions = f"""
    You are an Evidence Verifier for a fact-checking system.

    Your task is to verify a claim using ONLY the evidence provided to you.

    Do not use external knowledge.
    Do not assume facts that are not present in the evidence.

    A claim may contain multiple factual components.
    When this happens, consider the different components of the claim before assigning evidence relationships and the 
    final verdict.

    For each evidence item, determine its relationship to the claim:

    - SUPPORTS:
    The evidence provides factual information that supports the claim as a whole, or supports an important component without
    contradicting or leaving unresolved another essential component of the claim.

    - REFUTES:
    The evidence provides factual information that contradicts the claim as a whole, or directly contradicts an essential 
    component in a way that makes the overall claim false.

    - NEUTRAL:
    The evidence is related to the claim but does not provide enough information to support or refute it as a whole. 
    Use NEUTRAL when the evidence addresses only one component of a multi-part claim and leaves other essential
    components unresolved.

    Then determine the overall verdict:

    - SUPPORTED:
    The available evidence is sufficient to support the claim as a whole.

    - REFUTED:
    The available evidence is sufficient to refute the claim as a whole.

    - NOT_ENOUGH_EVIDENCE:
    The available evidence does not contain enough information to reach a justified conclusion about the claim.

    - CONFLICTING_EVIDENCE:
    The available evidence supports some important factual aspects of the claim while refuting, contradicting, or 
    substantially qualifying other important aspects, so that labeling the whole claim simply SUPPORTED or REFUTED would
    lose relevant information.

    Use this verdict also for cherry-picking cases, where the claim relies on factually supported information but 
    selectively combines, omits, or frames evidence in a way that produces a misleading overall conclusion.

    Do not automatically label a multi-part claim REFUTED simply because one component is contradicted if other important 
    components are supported.
    In such cases, consider whether CONFLICTING_EVIDENCE is more appropriate.

    Set evidence_sufficient to true when the provided evidence contains enough factual information to justify the selected
    verdict, including when the evidence is sufficient to establish that the case is CONFLICTING_EVIDENCE.

    Set evidence_sufficient to false when the appropriate verdict is
    NOT_ENOUGH_EVIDENCE.

    Assess every evidence item provided.

    Base the final verdict on the content of the evidence, not on retrieval similarity scores.

    {language_instruction}
    """

    response = client.responses.parse(
        model=model,
        instructions=instructions,
        input=f"""
    CLAIM:
    {claim}

    EVIDENCE:
    {evidence_text}
    """,
            text_format=VerificationResult,
        )

    return response.output_parsed

### 14. Diagnóstico final del componente de retrieval

El análisis de errores mostró que las claims 233 y 574 fueron clasificadas como
`NOT_ENOUGH_EVIDENCE` porque las evidencias necesarias para resolverlas no aparecieron
entre los fragmentos finalmente recuperados.

Antes de congelar el sistema se realiza una última inspección del retrieval para determinar
en qué etapa se pierde la evidencia relevante. El objetivo no es optimizar nuevos
hiperparámetros sobre estas claims, sino distinguir si la evidencia de referencia queda
fuera durante la preselección documental o si, estando disponible en el índice vectorial,
no alcanza las primeras posiciones de la búsqueda semántica.

Este análisis se realiza únicamente con fines diagnósticos y no modifica todavía la
configuración del pipeline.

¿Que queremos averiguar exactamente? Para 233 y 574 ya sabemos que Knowledge Store contiene evidencia buena pero el Top-10 final no contiene esa evidencia, pero nos falta saber dónde la perdemos: Knowledge Store completo, TF-IDF prefilter (si se activa), chunks que llegan a FAISS, candidate_k = 50, diversificación y Top-10. 

Hay tres posibilidades:

A -  El documento gold ni siquiera sobrevive al TF-IDF: problema de preselección documental

B -  Sobrevive y está en FAISS, pero queda fuera del Top-50: problema de búsqueda semántica/query

C -  Está dentro del Top-50, pero no llega al Top-10: problema de selección/diversificación final

Vamos a empezar por la claim 233

In [87]:
claim_id = 233
claim = train_df.loc[claim_id, "claim"]

#Ya conocemos de antes las URLs
gold_urls_233 = [
    "https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/",
    "https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history",
    "https://www.forbes.com/sites/jemimamcevoy/2020/06/08/14-days-of-protests-19-dead/",
]

Reconstruimos el retrieval sin llamar al LLM

In [88]:
knowledge_records = load_knowledge_records(
    knowledge_zip=knowledge_zip,
    claim_id=claim_id,
)

chunks, index, retrieval_info = build_claim_index(
    claim=claim,
    knowledge_records=knowledge_records,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
)

Corpus grande detectado: 150586 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 1115
Documentos únicos: 1112
Documentos finales: 100
Chunks iniciales: 150586
Chunks finales: 8933


Batches: 100%|██████████| 140/140 [34:30<00:00, 14.79s/it]


Vamos a ver si las URL están dentro de los chunks que llegaron a Faiss

In [89]:
for gold_url in gold_urls_233:

    matching_chunks = [
        chunk
        for chunk in chunks
        if chunk["url"] == gold_url
    ]

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    print("Chunks in final FAISS index:", len(matching_chunks))


GOLD URL:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/
Chunks in final FAISS index: 0

GOLD URL:
https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history
Chunks in final FAISS index: 0

GOLD URL:
https://www.forbes.com/sites/jemimamcevoy/2020/06/08/14-days-of-protests-19-dead/
Chunks in final FAISS index: 0


Este resultado significa que FAISS nunca pudo recuperar esas evidencias. Pero todavía hay dos posibilidades:

A -  Las URLs estaban en el Knowledge Store y se perdieron durante la preselección TF-IDF

B -  Las URLs del Knowledge Store no coinciden exactamente con las URLs gold que estamos comparando (archive URL, parámetros, http/https, etc.)

Así que el siguiente paso es comprobar si esas fuentes existen antes del build_claim_index().

In [90]:
for gold_url in gold_urls_233:

    matching_records = [
        record
        for record in knowledge_records
        if record.get("url") == gold_url
    ]

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    print("Records in original Knowledge Store:", len(matching_records))


GOLD URL:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/
Records in original Knowledge Store: 2

GOLD URL:
https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history
Records in original Knowledge Store: 2

GOLD URL:
https://www.forbes.com/sites/jemimamcevoy/2020/06/08/14-days-of-protests-19-dead/
Records in original Knowledge Store: 2


Con los resultados obtenido, tenemos que Knowledge Store sí contiene la evidencia gold pero la preselección documental la elimina y por tanto FAISS no llega a verla. En ese caso el problema de la 233 estaría antes del retrieval vectorial, concretamente en la selección previa de documentos, muy probablemente en el select_relevant_documents() con TF-IDF.

Aquí solo queremos confirmar que la URL existe y que tiene contenido textual utilizable

In [91]:
for gold_url in gold_urls_233:

    matching_records = [
        record
        for record in knowledge_records
        if record.get("url") == gold_url
    ]

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    for i, record in enumerate(matching_records, start=1):

        text_parts = record.get("url2text", [])

        print(f"\nRecord {i}")
        print("Source type:", record.get("type"))
        print("Text parts:", len(text_parts))

        if text_parts:
            print("Example:")
            print(text_parts[0][:500])


GOLD URL:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/

Record 1
Source type: gold
Text parts: 14
Example:
WASHINGTON — More than 700 law enforcement officers have been injured on the job during nationwide protests over the death of George Floyd — with nearly 300 of those among New York’s Finest, according to the Department of Justice and the NYPD.

Record 2
Source type: answer
Text parts: 14
Example:
WASHINGTON — More than 700 law enforcement officers have been injured on the job during nationwide protests over the death of George Floyd — with nearly 300 of those among New York’s Finest, according to the Department of Justice and the NYPD.

GOLD URL:
https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history

Record 1
Source type: gold
Text parts: 51
Example:
The costliest civil disorder in U.S. history: That's what insurance experts and city officials say the riots and demonstrations following the

Comprobamos directamente el TF-IDF

In [92]:
documents_233 = []

for record in knowledge_records:

    document = build_document(record)

    if document["text"].strip():
        documents_233.append(document)


selected_documents_233 = select_relevant_documents(
    claim=claim,
    documents=documents_233,
    top_n=100,
)

In [93]:
selected_urls_233 = {
    document["url"]
    for document in selected_documents_233
}

for gold_url in gold_urls_233:

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    print(
        "Selected by TF-IDF:",
        gold_url in selected_urls_233
    )


GOLD URL:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/
Selected by TF-IDF: False

GOLD URL:
https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history
Selected by TF-IDF: False

GOLD URL:
https://www.forbes.com/sites/jemimamcevoy/2020/06/08/14-days-of-protests-19-dead/
Selected by TF-IDF: False


Con este resultado vemos que aquí el problema no está en FAISS ni en el prompt, sino en la preselección TF-IDF que introdujimos para reducir el volumen de chunks. Vamos a averiguar en qué posiciones quedaron realmente esas tres fuentes dentro del ranking TF-IDF. Eso nos dirá si el problema se arreglaría razonablemente con 150, 200, 300 documentos, o si están muchísimo más abajo y el enfoque necesita otra cosa.

In [ ]:
document_texts = [
    document["text"]
    for document in documents_233
]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000
)

tfidf_matrix = vectorizer.fit_transform(
    [claim] + document_texts
)

claim_vector = tfidf_matrix[0]
document_vectors = tfidf_matrix[1:]

similarities = cosine_similarity(
    claim_vector,
    document_vectors
)[0]

ranked_indices = similarities.argsort()[::-1]


for gold_url in gold_urls_233:

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    matching_indices = [
        i
        for i, document in enumerate(documents_233)
        if document["url"] == gold_url
    ]

    for idx in matching_indices:

        rank = np.where(ranked_indices == idx)[0][0] + 1

        print("TF-IDF rank:", rank)
        print("TF-IDF score:", similarities[idx])


GOLD URL:
https://nypost.com/2020/06/08/more-than-700-officers-injured-in-george-floyd-protests-across-us/
TF-IDF rank: 365
TF-IDF score: 0.04782416188507958
TF-IDF rank: 364
TF-IDF score: 0.04782416188507958

GOLD URL:
https://www.foxnews.com/politics/george-floyd-protests-expensive-civil-disturbance-us-history
TF-IDF rank: 223
TF-IDF score: 0.07034084951328859
TF-IDF rank: 222
TF-IDF score: 0.07034084951328859

GOLD URL:
https://www.forbes.com/sites/jemimamcevoy/2020/06/08/14-days-of-protests-19-dead/
TF-IDF rank: 330
TF-IDF score: 0.05127622588384461
TF-IDF rank: 329
TF-IDF score: 0.05127622588384461


La inspección del ranking TF-IDF muestra que las fuentes gold de la claim 233
aparecen en posiciones relativamente bajas: aproximadamente 222–223 para FoxNews,
329–330 para Forbes y 364–365 para NYPost.

Por tanto, aumentar ligeramente el número de documentos preseleccionados no resolvería
el problema. Para garantizar la inclusión de todas las fuentes sería necesario conservar
más de 360 documentos, reduciendo de forma considerable el beneficio computacional de
la etapa de prefiltrado.

Este resultado sugiere que la limitación no depende únicamente del valor de
`top_n_documents`, sino del uso de la claim completa como única consulta léxica. En
afirmaciones multicomponente, distintas fuentes pueden aportar evidencia relevante para
componentes diferentes de la claim y presentar una similitud léxica limitada con la
afirmación completa.

En consecuencia, una posible mejora futura consistiría en generar subclaims o preguntas
de verificación y realizar la recuperación de evidencia de forma independiente para cada
componente.

Ahora para la claim 574 haremos practicamente lo mismo, pero de forma más directa.

In [95]:
claim_id = 574
claim = train_df.loc[claim_id, "claim"]

gold_urls_574 = [
    "https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query",
    "https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater",
]

In [96]:
knowledge_records_574 = load_knowledge_records(
    knowledge_zip=knowledge_zip,
    claim_id=claim_id,
)

In [97]:
for gold_url in gold_urls_574:

    matching_records = [
        record
        for record in knowledge_records_574
        if record.get("url") == gold_url
    ]

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    print("Records in original Knowledge Store:", len(matching_records))


GOLD URL:
https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query
Records in original Knowledge Store: 1

GOLD URL:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater
Records in original Knowledge Store: 1


In [98]:
documents_574 = []

for record in knowledge_records_574:

    document = build_document(record)

    if document["text"].strip():
        documents_574.append(document)

In [99]:
selected_documents_574 = select_relevant_documents(
    claim=claim,
    documents=documents_574,
    top_n=100,
)

In [100]:
selected_urls_574 = {
    document["url"]
    for document in selected_documents_574
}

for gold_url in gold_urls_574:

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    print(
        "Selected by TF-IDF:",
        gold_url in selected_urls_574
    )


GOLD URL:
https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query
Selected by TF-IDF: False

GOLD URL:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater
Selected by TF-IDF: False


In [102]:
document_texts = [
    document["text"]
    for document in documents_574
]

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000
)

tfidf_matrix = vectorizer.fit_transform(
    [claim] + document_texts
)

claim_vector = tfidf_matrix[0]
document_vectors = tfidf_matrix[1:]

similarities = cosine_similarity(
    claim_vector,
    document_vectors
)[0]

ranked_indices = similarities.argsort()[::-1]


for gold_url in gold_urls_574:

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    matching_indices = [
        i
        for i, document in enumerate(documents_574)
        if document["url"] == gold_url
    ]

    for idx in matching_indices:

        rank = np.where(ranked_indices == idx)[0][0] + 1

        print("TF-IDF rank:", rank)
        print("TF-IDF score:", similarities[idx])


GOLD URL:
https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query

GOLD URL:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater


In [104]:
for gold_url in gold_urls_574:

    matching_records = [
        record
        for record in knowledge_records_574
        if record.get("url") == gold_url
    ]

    print("\n" + "=" * 80)
    print("GOLD URL:")
    print(gold_url)

    for i, record in enumerate(matching_records, start=1):

        text_parts = record.get("url2text", [])

        print(f"\nRecord {i}")
        print("Source type:", record.get("type"))
        print("Text parts:", len(text_parts))

        if text_parts:
            print("Example:")
            print(text_parts[0][:500])


GOLD URL:
https://twitter.com/search?q=%22we%20wont%20be%20rebuilding%20Walmarts%20in%20cities%20without%20police%22%20(from%3AWalmart)&src=typed_query

Record 1
Source type: gold
Text parts: 0

GOLD URL:
https://web.archive.org/web/20200909211847/https://www.facebook.com/photo.php?fbid=10157247843507011&set=a.376354472010&type=3&theater

Record 1
Source type: gold
Text parts: 0


#### Diagnóstico de retrieval para la claim 574

La inspección de las fuentes gold muestra que las dos URLs utilizadas por AVeriTeC
están presentes en el Knowledge Store. Sin embargo, ambos registros presentan
`url2text` vacío.

Como el pipeline descarta los documentos sin contenido textual antes de la etapa de
chunking, estas fuentes no llegan a la preselección TF-IDF ni al índice FAISS. Por tanto,
el `Evidence Verifier` nunca recibe la evidencia que permite refutar la claim.

A diferencia de la claim 233, donde la evidencia relevante se pierde durante la
preselección TF-IDF, en la claim 574 la pérdida ocurre en una etapa anterior debido a la
ausencia de texto extraíble en las fuentes gold.

Este caso muestra una limitación adicional del sistema: la calidad del retrieval depende
también de la disponibilidad de contenido textual en las fuentes recuperadas, especialmente
en redes sociales, páginas archivadas o contenidos difíciles de extraer automáticamente.

Con esto ya tenemos dos tipos distintos de fallo de retrieval:

- 233: la evidencia existe y tiene texto, pero TF-IDF la elimina

- 574: la evidencia existe como URL, pero no tiene texto utilizable

Eso es bastante bueno para el análisis porque muestra que no hay un único `problema de retrieval`, sino fallos en etapas distintas.

#### Decisión final sobre el componente de retrieval

El análisis de las claims 233 y 574 muestra que los errores de recuperación no se
originan en un único punto del pipeline.

En la claim 233, las evidencias gold están disponibles y contienen texto, pero son
eliminadas por la preselección TF-IDF antes de llegar al índice vectorial. Además, sus
posiciones en el ranking TF-IDF son suficientemente bajas como para que aumentar
moderadamente `top_n_documents` no garantice su recuperación sin incrementar de forma
importante el coste computacional.

En la claim 574, las fuentes gold están presentes en el Knowledge Store, pero no contienen
texto extraíble en `url2text`, por lo que no pueden ser incorporadas al pipeline de
retrieval textual.

Estos resultados indican limitaciones estructurales relacionadas tanto con la
preselección basada en la claim completa como con la disponibilidad de contenido textual
de las fuentes. Por este motivo, no se realizan nuevos ajustes de retrieval antes de la
evaluación sobre `dev`.

La configuración actual se mantiene congelada y las mejoras basadas en generación de
subclaims, preguntas de verificación, recuperación multi-query o mecanismos alternativos
de extracción de contenido se consideran extensiones futuras del sistema.

### 15. Congelación de la configuración antes de la evaluación en dev
#### (Cierre de la fase de desarrollo)

Tras las pruebas realizadas sobre el conjunto de entrenamiento, se fija la configuración
del sistema que será utilizada durante la evaluación posterior.

La configuración congelada incluye:

- preselección TF-IDF de documentos cuando el número de chunks supera el límite definido;
- `top_n_documents=100`;
- embeddings `BAAI/bge-small-en-v1.5`;
- índice FAISS con similitud basada en producto interno sobre embeddings normalizados;
- recuperación inicial de `candidate_k=50`;
- selección final de 10 evidencias con control de diversidad por documento;
- versión V2 del prompt del `Evidence Verifier`;
- las cuatro clases normalizadas de AVeriTeC.

A partir de este punto no se realizarán nuevos ajustes utilizando las etiquetas o errores
del conjunto `dev`. Este conjunto se utilizará exclusivamente como evaluación held-out
del sistema congelado.

### 16. Evaluación final sobre el conjunto dev

Una vez finalizada la fase de desarrollo y congelada la configuración del sistema,
se utiliza el conjunto `dev` de AVeriTeC como conjunto held-out para evaluar su
capacidad de generalización sobre afirmaciones no utilizadas durante el proceso de
ajuste.

`Conjunto held-out se refiere al conjunto dejado aparte.`

`holdout: Se refiere a la práctica de dividir un conjunto de datos en dos subconjuntos distintos: uno para entrenar un modelo y otro para evaluar su rendimiento`

Durante esta fase no se realizarán modificaciones en el prompt, el componente de
retrieval ni los hiperparámetros del pipeline a partir de los resultados obtenidos.

La evaluación comparará las predicciones generadas por el sistema con las etiquetas
gold de AVeriTeC y se calcularán métricas globales y por clase.

In [37]:
DEV_KNOWLEDGE_FILE = (
    "data_store/knowledge_store/"
    "dev_knowledge_store.zip"
)

knowledge_zip = hf_hub_download(
    repo_id="chenxwh/AVeriTeC",
    filename=DEV_KNOWLEDGE_FILE,
    local_dir=KNOWLEDGE_DIR
)

print(knowledge_zip)

C:\Users\natal\OneDrive\Documentos\UCM - Big data, ML and IA\MODULOS\TFM\TFM_Natalia_De_Oliveira_AgenteFakeNews\Data\averitec\raw\knowledge_store\data_store\knowledge_store\dev_knowledge_store.zip


In [38]:
dev_knowledge_zip = (
    KNOWLEDGE_DIR
    / "data_store"
    / "knowledge_store"
    / "dev_knowledge_store.zip"
)

print(dev_knowledge_zip)
print("Exists:", dev_knowledge_zip.exists())

..\Data\averitec\raw\knowledge_store\data_store\knowledge_store\dev_knowledge_store.zip
Exists: True


Viendo la distribución de las clases en Dev:

In [39]:
dev_df["label"].value_counts()

label
Refuted                               305
Supported                             122
Conflicting Evidence/Cherrypicking     38
Not Enough Evidence                    35
Name: count, dtype: int64

In [109]:
dev_df["label"].value_counts(normalize=True)

label
Refuted                               0.610
Supported                             0.244
Conflicting Evidence/Cherrypicking    0.076
Not Enough Evidence                   0.070
Name: proportion, dtype: float64

Esto nos confirma que dev está bastante desbalanceado, por eso no usaremos una muestra aleatoria simple de 40 claims, porque probablemente saldrían muchas Refuted y muy pocas de las clases minoritarias. Para nuestro objetivo de evaluar el comportamiento por clase, mantendremos la idea de una muestra estratificada y balanceada:

10 Refuted <br>
10 Supported <br>
10 Conflicting Evidence/Cherrypicking <br>
10 Not Enough Evidence

#### Selección de la muestra de evaluación

El conjunto `dev` contiene 500 afirmaciones y presenta una distribución de clases
desbalanceada, con una mayoría de ejemplos `Refuted` (61 %) y una representación
considerablemente menor de `Conflicting Evidence/Cherrypicking` y
`Not Enough Evidence`.

Dado el coste computacional del pipeline completo, se utiliza una muestra estratificada
balanceada de 40 claims, seleccionando 10 ejemplos de cada una de las cuatro clases.

Esta estrategia permite evaluar de forma comparable el comportamiento del sistema en
todas las categorías y evita que las métricas estén dominadas por la clase mayoritaria.

In [40]:
dev_evaluation_sample = (
    dev_df
    .groupby("label", group_keys=False)
    .sample(n=10, random_state=42)
)

dev_evaluation_sample["label"].value_counts()

label
Conflicting Evidence/Cherrypicking    10
Not Enough Evidence                   10
Refuted                               10
Supported                             10
Name: count, dtype: int64

Vamos a ejecutar las 40 claims con la configuración ya congelada:

In [41]:
DEV_RESULTS_PATH = PROCESSED_DIR / "dev_evaluation_results.csv"

In [44]:
dev_results_df = evaluate_averitec_sample(
    evaluation_sample=dev_evaluation_sample,
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
    output_path=DEV_RESULTS_PATH,
)

Resultados anteriores encontrados: 40 claims.

Claim 457 ya evaluada. Se omite.

Claim 482 ya evaluada. Se omite.

Claim 43 ya evaluada. Se omite.

Claim 115 ya evaluada. Se omite.

Claim 427 ya evaluada. Se omite.

Claim 370 ya evaluada. Se omite.

Claim 50 ya evaluada. Se omite.

Claim 389 ya evaluada. Se omite.

Claim 360 ya evaluada. Se omite.

Claim 148 ya evaluada. Se omite.

Claim 209 ya evaluada. Se omite.

Claim 210 ya evaluada. Se omite.

Claim 68 ya evaluada. Se omite.

Claim 153 ya evaluada. Se omite.

Claim 394 ya evaluada. Se omite.

Claim 233 ya evaluada. Se omite.

Claim 183 ya evaluada. Se omite.

Claim 76 ya evaluada. Se omite.

Claim 223 ya evaluada. Se omite.

Claim 192 ya evaluada. Se omite.

Claim 333 ya evaluada. Se omite.

Claim 52 ya evaluada. Se omite.

Claim 23 ya evaluada. Se omite.

Claim 80 ya evaluada. Se omite.

Claim 91 ya evaluada. Se omite.

Claim 316 ya evaluada. Se omite.

Claim 120 ya evaluada. Se omite.

Claim 193 ya evaluada. Se omite.

Claim 268

In [45]:
y_true = dev_results_df["gold"]
y_pred = dev_results_df["predicted"]

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)

print("\nClassification report:")
print(classification_report(y_true, y_pred))

Accuracy: 0.4
Macro-F1: 0.3630555555555556

Classification report:
                      precision    recall  f1-score   support

CONFLICTING_EVIDENCE       0.00      0.00      0.00        10
 NOT_ENOUGH_EVIDENCE       0.36      0.50      0.42        10
             REFUTED       0.40      0.60      0.48        10
           SUPPORTED       0.62      0.50      0.56        10

            accuracy                           0.40        40
           macro avg       0.35      0.40      0.36        40
        weighted avg       0.35      0.40      0.36        40



In [46]:
labels = [
    "SUPPORTED",
    "REFUTED",
    "NOT_ENOUGH_EVIDENCE",
    "CONFLICTING_EVIDENCE",
]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels,
)

cm

array([[5, 2, 2, 1],
       [0, 6, 3, 1],
       [2, 2, 5, 1],
       [1, 5, 4, 0]])

#### Resultados sobre el conjunto dev

La evaluación sobre una muestra estratificada de 40 claims obtuvo una
accuracy de 0.40 y un macro-F1 de 0.36.

El comportamiento varía de forma considerable entre clases. El sistema obtiene mejores
resultados en `SUPPORTED` y `REFUTED`, mientras que presenta dificultades especialmente
relevantes en `CONFLICTING_EVIDENCE`, clase para la que no se obtiene ninguna predicción
correcta en la muestra evaluada.

La matriz de confusión muestra que las claims gold `CONFLICTING_EVIDENCE` tienden a ser
clasificadas principalmente como `REFUTED` o `NOT_ENOUGH_EVIDENCE`, lo que indica una
limitación del sistema para distinguir entre una contradicción directa y situaciones en
las que la evidencia apoya y refuta distintos aspectos de una misma claim o introduce
matices sustanciales.

Dado que `dev` se utiliza como conjunto held-out, estos resultados no se emplearán para
realizar nuevos ajustes del prompt o del componente de retrieval. Se utilizarán como
medida final de generalización del componente factual y como base para el análisis de
limitaciones y posibles mejoras futuras.

### 16.1 Análisis descriptivos de los errores de `dev`

In [47]:
dev_errors_df = dev_results_df[
    dev_results_df["correct"] == False
].copy()

dev_errors_df[
    [
        "claim_id",
        "claim",
        "gold",
        "predicted",
        "evidence_sufficient",
    ]
]

,claim_id,claim,gold,predicted,evidence_sufficient
0,457,The Radio Act in Canada makes it a crime to li...,CONFLICTING_EVIDENCE,REFUTED,True
1,482,"Donald trump said: ""Joe Biden recently raised ...",CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,False
2,43,Donald Trump said that $15 an hour is too much...,CONFLICTING_EVIDENCE,REFUTED,True
3,115,More people who wear masks become sick with CO...,CONFLICTING_EVIDENCE,REFUTED,True
4,427,U.S. citizens should show up at polling places...,CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,False
5,370,Government of India has imposed taxes on all s...,CONFLICTING_EVIDENCE,REFUTED,True
6,50,Twitter now putting warnings on tweets that ar...,CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,False
7,389,The cardinal and unforgivable sin of a French ...,CONFLICTING_EVIDENCE,SUPPORTED,True
8,360,These unlicensed vaccines will be administered...,CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,False
9,148,U.S. President Donald Trump has suspended all ...,CONFLICTING_EVIDENCE,REFUTED,True


In [49]:
len(dev_errors_df)

24

In [48]:
(
    dev_errors_df
    .groupby(["gold", "predicted"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

,gold,predicted,count
1,CONFLICTING_EVIDENCE,REFUTED,5
0,CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,4
7,REFUTED,NOT_ENOUGH_EVIDENCE,3
10,SUPPORTED,REFUTED,2
4,NOT_ENOUGH_EVIDENCE,REFUTED,2
9,SUPPORTED,NOT_ENOUGH_EVIDENCE,2
5,NOT_ENOUGH_EVIDENCE,SUPPORTED,2
2,CONFLICTING_EVIDENCE,SUPPORTED,1
3,NOT_ENOUGH_EVIDENCE,CONFLICTING_EVIDENCE,1
6,REFUTED,CONFLICTING_EVIDENCE,1


Tras la evaluación se identificaron 24 predicciones incorrectas de las
40 claims evaluadas. Los errores no se distribuyen uniformemente entre las clases.

La principal dificultad se observa en `CONFLICTING_EVIDENCE`: ninguna de las diez
claims de esta categoría fue clasificada correctamente. Cinco fueron clasificadas
como `REFUTED`, cuatro como `NOT_ENOUGH_EVIDENCE` y una como `SUPPORTED`.

También se observan errores `REFUTED: NOT_ENOUGH_EVIDENCE`, así como confusiones
entre `SUPPORTED`, `REFUTED` y `NOT_ENOUGH_EVIDENCE`.

Para estudiar las posibles causas sin realizar nuevos ajustes sobre el conjunto, se selecciona un pequeño conjunto de errores representativos. El análisis será exclusivamente descriptivo y no se utilizará para modificar la configuración congelada del sistema. Vamos a seleccionar claims en las que vamos a poder revisar el retrieval y el evidence verifier.

In [50]:
representative_dev_ids = [
    457,  # CONFLICTING:  REFUTED, evidence_sufficient=True
    482,  # CONFLICTING:  NEI, evidence_sufficient=False
    333,  # REFUTED: NEI, evidence_sufficient=False
    68,   # NEI: REFUTED, evidence_sufficient=True
    57,   # SUPPORTED: REFUTED, evidence_sufficient=True
    93,   # SUPPORTED: NEI, evidence_sufficient=False
]

representative_dev_errors = dev_errors_df[
    dev_errors_df["claim_id"].isin(representative_dev_ids)
].copy()

representative_dev_errors[
    [
        "claim_id",
        "claim",
        "gold",
        "predicted",
        "evidence_sufficient",
        "explanation",
    ]
]

,claim_id,claim,gold,predicted,evidence_sufficient,explanation
0,457,The Radio Act in Canada makes it a crime to li...,CONFLICTING_EVIDENCE,REFUTED,True,The evidence indicates that false or misleadin...
1,482,"Donald trump said: ""Joe Biden recently raised ...",CONFLICTING_EVIDENCE,NOT_ENOUGH_EVIDENCE,False,The evidence supports the limited fact that Bi...
12,68,Joe Biden said 'If Your 8-Year-Old Child Says ...,NOT_ENOUGH_EVIDENCE,REFUTED,True,The evidence documents Biden's town-hall remar...
20,333,Development Control Department of the Abuja Me...,REFUTED,NOT_ENOUGH_EVIDENCE,False,The evidence establishes that AMMC/FCTA develo...
32,57,While serving as Town Supervisor on Grand Isla...,SUPPORTED,REFUTED,True,The evidence consistently places Nate McMurray...
33,93,US Judge Amy Coney Barrett graduated at the to...,SUPPORTED,NOT_ENOUGH_EVIDENCE,False,The evidence consistently supports that Amy Co...


Vamos a ver la claim `457`. Con esta claim queremos responder a lo siguiente: ¿El retrieval recuperó información que mostraba support + contradiction, pero el verifier la redujo a REFUTED?
o ¿el retrieval ya perdió la parte de la evidencia que hacía que el gold fuera `CONFLICTING_EVIDENCE`?

In [51]:
claim_id = 457

row_457 = dev_df.loc[claim_id]

result_457 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_457["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 171628 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 973
Documentos únicos: 972
Documentos finales: 100
Chunks iniciales: 171628
Chunks finales: 9848


Batches: 100%|██████████| 154/154 [29:16<00:00, 11.41s/it]


In [52]:
print("Claim:")
print(row_457["claim"])

print("\nGold:")
print(normalize_averitec_label(row_457["label"]))

print("\nPredicted:")
print(result_457["verification"].verdict)

print("\nEvidence sufficient:")
print(result_457["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_457["verification"].explanation)

Claim:
The Radio Act in Canada makes it a crime to lie to the public via airwaves.

Gold:
CONFLICTING_EVIDENCE

Predicted:
REFUTED

Evidence sufficient:
True

Explanation:
The evidence indicates that Canadian broadcast policy may prohibit false or misleading news, but the cited account says such false-news provisions are regulations rather than criminal offences after the relevant Criminal Code provision was ruled unconstitutional. Therefore, the assertion that the Radio Act makes lying to the public via airwaves a crime is refuted.


In [53]:
for i, evidence in enumerate(result_457["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://www.snopes.com/fact-check/canadian-fox/
Source type: question_duplicate
Score: 0.7134578227996826
Text: Since at least 2011, rumors have circulated claiming the Fox News television channel has been banned in Canada due to their running afoul of Canadian Radio-Television and Telecommunications Commission (CRTC) prohibitions that make it "illegal to broadcast lies and label it news": One prominent example of this rumor stated, for example, that: America's middle class battles for its survival on the Wisconsin barricades — against various Koch Oil surrogates and the corporate toadies at Fox News — fans of enlightenment, democracy and justice can take comfort from a significant victory north of the Wisconsin border. Fox News will not be moving into Canada after all! The reason: Canadian regulators announced last week they would reject efforts by Canada's right-wing Prime Minister, Stephen Harper, to repeal a law that forbids lying on broadcast news. Canada'

##### Claim 457 — `CONFLICTING_EVIDENCE`: `REFUTED`

La claim afirma que la legislación canadiense convierte en delito mentir al público
a través de las ondas de radio.

La evidencia recuperada contiene información relevante para los distintos componentes
de la afirmación. Por una parte, se recupera información que indica que la regulación
canadiense de radiodifusión contempla restricciones sobre noticias falsas o engañosas.
Por otra, una de las evidencias señala explícitamente que las disposiciones relativas
a noticias falsas dejaron de constituir una infracción penal tras declararse
inconstitucional la correspondiente disposición del código penal, permaneciendo como
regulaciones.

Por tanto, el retrieval proporciona información suficiente para identificar tanto una
base factual parcialmente respaldada como una caracterización incorrecta de dicha
regulación como delito.

El `Evidence Verifier` identifica ambos aspectos en su explicación, pero asigna finalmente
`REFUTED`. El error se atribuye principalmente al razonamiento del verifier, que da
prioridad al componente contradicho (`crime`) y reduce el veredicto global a `REFUTED`,
en lugar de conservar la combinación de elementos respaldados y contradichos mediante
`CONFLICTING_EVIDENCE`.

En este caso, `evidence_sufficient=True` resulta coherente: la evidencia recuperada es
suficiente para alcanzar una conclusión, pero el error se produce en la asignación de
la clase final.

Vamos a ver la Claim `482`. Con esta claim queremos saber si ¿faltó recuperar evidencia suficiente
para mostrar el conflicto? o ¿sí estaba la evidencia, pero el verifier no supo interpretarla?

In [54]:
claim_id = 482

row_482 = dev_df.loc[claim_id]

result_482 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_482["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 43689 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 794
Documentos únicos: 793
Documentos finales: 100
Chunks iniciales: 43689
Chunks finales: 2773


Batches: 100%|██████████| 44/44 [05:23<00:00,  7.34s/it]


In [55]:
print("Claim:")
print(row_482["claim"])

print("\nGold:")
print(normalize_averitec_label(row_482["label"]))

print("\nPredicted:")
print(result_482["verification"].verdict)

print("\nEvidence sufficient:")
print(result_482["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_482["verification"].explanation)

Claim:
Donald trump said: "Joe Biden recently raised his hand on the debate stage and promised he was going to give it away, your health care dollars to illegal immigrants, which is going to bring massive number of immigrants into our country."

Gold:
CONFLICTING_EVIDENCE

Predicted:
NOT_ENOUGH_EVIDENCE

Evidence sufficient:
False

Explanation:
The evidence confirms an important underlying event: Biden raised his hand during a debate in response to a question about coverage for undocumented immigrants. It also documents other Trump comments on that event. However, none of the provided evidence contains or otherwise verifies the specific statement attributed to Trump, particularly the wording about giving away health-care dollars and causing a massive number of immigrants to enter the country. Therefore, the evidence is insufficient to verify the claim that Trump said this quotation.


In [56]:
for i, evidence in enumerate(result_482["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://www.cnbc.com/2019/07/05/joe-biden-vows-to-bring-back-obamacare-individual-mandate-penalty.html
Source type: question
Score: 0.8146544694900513
Text: Trump said. "How about taking care of American Citizens first!? That's the end of that race!" However, Trump's rhetoric was not surprising to Biden. "This is part of what Trump is playing on," he said, playing on people's fears of having open borders and people flowing into the U.S. Biden said providing health care to people who are sick "is just common decency."

--- EVIDENCE 2 ---
URL: https://kffhealthnews.org/morning-breakout/all-democratic-candidates-support-health-care-for-undocumented-immigrants-we-do-ourselves-no-favors-when-millions-cant-access-care/
Source type: question
Score: 0.8076920509338379
Text: if he would provide federally supported health coverage to undocumented immigrants, Biden quickly corrected her. You cannot let people who are sick, no matter where they come from, no matter what th

##### Claim 482 — `CONFLICTING_EVIDENCE`: `NOT_ENOUGH_EVIDENCE`

La claim atribuye a Donald Trump una declaración sobre la posición de Joe Biden
respecto a la cobertura sanitaria de inmigrantes indocumentados.

Las evidencias recuperadas permiten confirmar varios componentes relevantes de la
afirmación. Se documenta que Biden levantó la mano durante un debate ante una pregunta
sobre cobertura sanitaria para inmigrantes indocumentados y también se recuperan
declaraciones de Trump reaccionando a dicho episodio. Asimismo, otras evidencias
matizan la interpretación de la posición de Biden, indicando que permitir el acceso
o la compra de cobertura sanitaria no equivale necesariamente a proporcionar atención
sanitaria pública gratuita.

Sin embargo, las evidencias recuperadas no contienen de forma explícita la cita completa
atribuida a Trump en la claim. El `Evidence Verifier` centra su decisión en esta ausencia
y devuelve `NOT_ENOUGH_EVIDENCE`, estableciendo `evidence_sufficient=False`.

El error se considera mixto. Aunque el retrieval no recupera la formulación exacta de
la cita, sí proporciona información suficiente sobre varios de sus componentes
materiales y sobre la posible tergiversación de la posición de Biden. El verifier no
integra estas evidencias parciales como un posible caso de información factual combinada
con una caracterización engañosa, lo que habría favorecido `CONFLICTING_EVIDENCE`.

Vamos a ver la claim `333`. Aqui queremos comprobar si el error viene realmente del retrieval. Queremos saber si ¿La evidencia que permitía refutar la claim estaba en el Knowledge Store pero no llegó al Top-10? o ¿Sí llegó al Top-10 y aun así el verifier no la interpretó bien?

In [57]:
claim_id = 333

row_333 = dev_df.loc[claim_id]

result_333 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_333["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 87143 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 488
Documentos únicos: 488
Documentos finales: 100
Chunks iniciales: 87143
Chunks finales: 832


Batches: 100%|██████████| 13/13 [01:42<00:00,  7.85s/it]


In [58]:
print("Claim:")
print(row_333["claim"])

print("\nGold:")
print(normalize_averitec_label(row_333["label"]))

print("\nPredicted:")
print(result_333["verification"].verdict)

print("\nEvidence sufficient:")
print(result_333["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_333["verification"].explanation)

Claim:
Development Control Department of the Abuja Metropolitan Management Council (AMMC), an agency of the Federal Capital Territory Administration (FCTA)of Nigeria, in 2020 marked 37 Estates for demolition.

Gold:
REFUTED

Predicted:
NOT_ENOUGH_EVIDENCE

Evidence sufficient:
False

Explanation:
The evidence supports that AMMC includes or works through a Development Control Department and that FCTA/AMMC authorities have conducted demolition-related marking and demolition activities, including a 2020 case involving one community. However, none of the evidence states that the Development Control Department marked 37 estates for demolition in 2020. The claim's central numerical and event-specific assertion is therefore unverified.


In [59]:
for i, evidence in enumerate(result_333["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://dailypost.ng/2018/01/31/abuja-master-plan-fcta-vows-demolish-750-buildings/
Source type: provenance
Score: 0.8339827060699463
Text: Abuja Master Plan: FCTA vows to demolish 750 buildings The Federal Capital Territory Administration (FCTA) has vowed to demolish 750 illegal building in the nation’s capital in order to restore its Master Plan. The Coordinator, Abuja Metropolitan Management Council, AMMC, Shuaibu Umar, made this known while speaking at a press parley held in Abuja, on Wednesday. According to him, the 750 illegal shanties marked for demolition were located in Lugbe District. Umar also revealed that additional structures that breached the permitted 30 meters proximity to High Tension Lines as well as those that stood in the way of Transmission Company of Nigeria (TCN)’s work on a new transmission station will be removed. The Coordinator maintained that the dream of making Abuja a smart city is attainable; adding that efforts are being made to

##### Claim 333 — `REFUTED`: `NOT_ENOUGH_EVIDENCE`

La claim afirma que el Development Control Department del AMMC marcó 37 estates
para demolición en 2020.

Las evidencias recuperadas contienen información relevante sobre el papel del AMMC
y del Department of Development Control, así como distintos casos de marcado y demolición
de estructuras en Abuja. Sin embargo, ninguna de las evidencias recuperadas menciona
específicamente que se marcaran 37 estates para demolición en 2020.

El `Evidence Verifier` devuelve `NOT_ENOUGH_EVIDENCE` y establece
`evidence_sufficient=False`, una decisión coherente con la información que recibió,
ya que el componente numérico y temporal central de la claim no aparece verificado
ni contradicho en el Top-10 recuperado.

Dado que la etiqueta gold es `REFUTED`, el error se atribuye principalmente al componente
de retrieval: la evidencia necesaria para refutar la afirmación específica no llegó
al conjunto final de evidencias proporcionadas al verifier.

Vamos a ver la claim `68`. Aqui queremos ver un tipo de error diferente al de la claim 333. Aqui queremos ver si ¿el retrieval recuperó evidencia que realmente contradice la claim, o el verifier interpretó como contradicción algo que solo era parcial, indirectamente relacionado o insuficiente?

In [60]:
claim_id = 68

row_68 = dev_df.loc[claim_id]

result_68 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_68["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 59678 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 687
Documentos únicos: 687
Documentos finales: 100
Chunks iniciales: 59678
Chunks finales: 5342


Batches: 100%|██████████| 84/84 [16:14<00:00, 11.60s/it]


In [61]:
print("Claim:")
print(row_68["claim"])

print("\nGold:")
print(normalize_averitec_label(row_68["label"]))

print("\nPredicted:")
print(result_68["verification"].verdict)

print("\nEvidence sufficient:")
print(result_68["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_68["verification"].explanation)

Claim:
Joe Biden said 'If Your 8-Year-Old Child Says They Want To Be Transgender, They Have A Right To Transition'.

Gold:
NOT_ENOUGH_EVIDENCE

Predicted:
REFUTED

Evidence sufficient:
True

Explanation:
The evidence establishes that Biden discussed an eight- or ten-year-old saying they wanted to be transgender and called for zero discrimination and equal rights. However, the direct transcript does not show him saying that an eight-year-old who says they want to be transgender has a right to transition. Commentary that interprets his remarks as support for transition does not establish the claimed quotation or its asserted right to transition.


In [62]:
for i, evidence in enumerate(result_68["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://www.prnewswire.com/news-releases/why-joe-biden-is-wrong-on-gender-transition-for-eight-year-olds-frc-action-explains-301154164.html
Source type: question_duplicate
Score: 0.8444463610649109
Text: Why Joe Biden Is Wrong on Gender Transition for Eight-Year-Olds: FRC Action Explains WASHINGTON, Oct. 16, 2020 /PRNewswire/ -- Last night, during a townhall on ABC, Joe Biden endorsed gender transition for eight-year-old children. Biden seemed to be promoting the unfounded belief that all children with "gender dysphoria" are innately and immutably "transgender," by referring sarcastically to "the idea that an 8-year-old or a 10-year-old, decides, you know, I want to be transgender -- that's what I think I'd like to be, make my life a lot easier." In response, Family Research Council released Friday a new issue analysis publication that details the scientific evidence of long-term harm of gender transition procedures on minors. Peter Sprigg, Senior Fellow for Po

##### Claim 68 — `NOT_ENOUGH_EVIDENCE`: `REFUTED`

La claim atribuye a Joe Biden una declaración según la cual un niño de ocho años que
manifieste querer ser transgénero tendría derecho a realizar una transición.

El retrieval recupera evidencias directamente relacionadas con el episodio, incluyendo
la transcripción del town hall en la que Biden habla de un niño de ocho o diez años que
dice querer ser transgénero y defiende la ausencia de discriminación y la igualdad de
derechos.

Sin embargo, la formulación específica relativa a un supuesto "right to transition"
no aparece explícitamente en la transcripción recuperada. Algunas fuentes secundarias
interpretan sus declaraciones como apoyo a la transición de menores, pero esta
interpretación no equivale a demostrar que Biden pronunciara la afirmación atribuida
en la claim.

El `Evidence Verifier` devuelve `REFUTED` y `evidence_sufficient=True`, interpretando
la ausencia de la cita exacta en la evidencia recuperada como una refutación. Dado que
esta ausencia no demuestra necesariamente que la declaración nunca se produjera o que
no exista una formulación equivalente en otra fuente, el error se atribuye
principalmente al verifier, que alcanza una conclusión más fuerte de la que permite
la evidencia disponible.

Vamos a analizar la claim `57`. Aqui queremos analizar si ¿el retrieval recuperó evidencia que realmente apoyaba la claim pero el verifier la interpretó al revés? o ¿el retrieval trajo evidencia contradictoria / incompleta que llevó al verifier a REFUTED?

In [63]:
claim_id = 57

row_57 = dev_df.loc[claim_id]

result_57 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_57["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 99616 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 658
Documentos únicos: 658
Documentos finales: 100
Chunks iniciales: 99616
Chunks finales: 1426


Batches: 100%|██████████| 23/23 [02:37<00:00,  6.84s/it]


In [64]:
print("Claim:")
print(row_57["claim"])

print("\nGold:")
print(normalize_averitec_label(row_57["label"]))

print("\nPredicted:")
print(result_57["verification"].verdict)

print("\nEvidence sufficient:")
print(result_57["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_57["verification"].explanation)

Claim:
While serving as Town Supervisor on Grand Island, Nebraska, US Nate McMurray voted to raise taxes on homeowners.

Gold:
SUPPORTED

Predicted:
REFUTED

Evidence sufficient:
True

Explanation:
The evidence repeatedly identifies Nate McMurray as Town Supervisor of Grand Island, New York/Western New York, not Grand Island, Nebraska. This directly contradicts an essential factual component of the claim. None of the evidence establishes that he voted to raise taxes on homeowners, but the claim as written is nevertheless refuted by its incorrect location.


In [65]:
for i, evidence in enumerate(result_57["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://ballotpedia.org/Nate_McMurray
Source type: question
Score: 0.7750569581985474
Text: in business development to Grand Island. He has championed the State’s plan for a major hike and bike trail along the waterfront, secured Grand Island as the location for the State’s Western New York Visitors’ Center, and spearheaded removal of the much-despised toll barriers at the North and South Grand Island Bridges... Nate’s experience as Grand Island Town Supervisor has taught him important lessons. “We are barely scratching the surface of Western New York’s potential,” Nate says. “We have amazing people, wonderful natural resources, and unbelievable potential. With the right leadership and some hard work, there is no limit to what we can do!” With his wide-ranging experience, deep love for Western New York, and vision for our future: Nathan McMurray is the perfect fit for the 27th District. The following is an example of an ad from McMurray's 2018 election campaign

##### Claim 57 — `SUPPORTED`: `REFUTED`

La claim afirma que Nate McMurray, durante su etapa como Town Supervisor de Grand Island,
votó para aumentar los impuestos sobre propietarios, aunque sitúa dicha localidad en
Nebraska.

Las evidencias recuperadas identifican de forma consistente a McMurray como Town
Supervisor de Grand Island, New York, por lo que contradicen el componente geográfico
de la afirmación. Sin embargo, el Top-10 no aporta evidencia clara sobre el componente
principal relativo a una votación para aumentar impuestos a los propietarios.

El `Evidence Verifier` considera la discrepancia geográfica como una contradicción
suficiente para clasificar la claim como `REFUTED` y establece
`evidence_sufficient=True`.

El error se considera mixto. Por una parte, el retrieval recupera principalmente
información biográfica y geográfica, pero no evidencia específica sobre la supuesta
votación fiscal. Por otra, el verifier da un peso decisivo al error de localización y
utiliza ese componente para refutar la claim completa, en lugar de distinguir entre
un detalle geográfico incorrecto y el núcleo sustantivo de la afirmación.

Analizamos ahora la claim `93`, la última de nuestro análisis. Aqui queremos comprobar si ¿la evidencia de apoyo estaba en el Knowledge Store pero no llegó al Top-10? o ¿sí llegó al Top-10 y aun así el verifier no la consideró suficiente?

In [66]:
claim_id = 93

row_93 = dev_df.loc[claim_id]

result_93 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_93["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 41313 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 645
Documentos únicos: 645
Documentos finales: 100
Chunks iniciales: 41313
Chunks finales: 1412


Batches: 100%|██████████| 23/23 [02:48<00:00,  7.32s/it]


In [67]:
print("Claim:")
print(row_93["claim"])

print("\nGold:")
print(normalize_averitec_label(row_93["label"]))

print("\nPredicted:")
print(result_93["verification"].verdict)

print("\nEvidence sufficient:")
print(result_93["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_93["verification"].explanation)

Claim:
US Judge Amy Coney Barrett graduated at the top of her law school class at Notre Dame Law School

Gold:
SUPPORTED

Predicted:
SUPPORTED

Evidence sufficient:
True

Explanation:
The evidence sufficiently supports the claim. Multiple items establish that Amy Coney Barrett graduated from Notre Dame Law School, and Evidence 6 specifically reports that she graduated summa cum laude and received the Hoynes Prize for the best overall record in scholarship, deportment, and achievement. Evidence 2 likewise identifies the Hoynes Prize as the law school's top honor.


In [68]:
for i, evidence in enumerate(result_93["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://news.nd.edu/news/notre-dame-law-school-professor-barrett-nominated-to-us-supreme-court/
Source type: question
Score: 0.8889598846435547
Text: Judge Amy Coney Barrett, professor of law at the University of Notre Dame and a 1997 graduate of Notre Dame Law School, was nominated today to the Supreme Court of the United States to fill the vacancy created by the death of Associate Justice Ruth Bader Ginsburg. She is the first Notre Dame graduate and faculty member to be nominated to serve on the nation’s highest court. “The same impressive intellect, character and temperament that made Judge Barrett a successful nominee for the U.S. Court of Appeals will serve her and the nation equally well as a Justice of the United States Supreme Court,” Notre Dame President Rev. John I. Jenkins, C.S.C., said. “An alumna and a faculty member of Notre Dame Law School, Judge Barrett has epitomized the University’s commitment to teaching, scholarship, justice and service to s

#### Variabilidad entre ejecuciones del Evidence Verifier

Durante el análisis descriptivo de errores se observó que la claim 93, clasificada como
`NOT_ENOUGH_EVIDENCE` durante la evaluación inicial (held-out), produjo `SUPPORTED` al ejecutarse
posteriormente de forma individual con la misma configuración general del pipeline.

La segunda ejecución recuperó evidencia suficiente para respaldar la afirmación y
coincidió con la etiqueta gold. Esta diferencia pone de manifiesto una posible
variabilidad entre ejecuciones asociada al componente basado en LLM.

Para preservar la validez de la evaluación, las métricas reportadas se mantienen a partir
de la ejecución held-out original y no se sustituyen predicciones individuales obtenidas
en ejecuciones posteriores. Las nuevas ejecuciones se utilizan únicamente con fines
diagnósticos.

Debido al resultado obtenido con la claim 93 de nuestro análisis, no podemos atribuirle un fallo estructural concreto, por lo tanto vamos a analizar ahora la claim 305 como sustitución de la claim 93. 

En la evaluación held-out, la claim 305 obtuvo `SUPPORTED` como etiqueta gold, mientras
que el sistema predijo `NOT_ENOUGH_EVIDENCE` con `evidence_sufficient=False`.

El objetivo de este análisis es determinar si la evidencia necesaria para apoyar la claim
no fue recuperada por el componente de retrieval o si, por el contrario, sí estaba presente
en el Top-10 pero no fue considerada suficiente por el `Evidence Verifier`.

In [69]:
claim_id = 305

row_305 = dev_df.loc[claim_id]

result_305 = verify_averitec_claim(
    claim_id=claim_id,
    claim=row_305["claim"],
    knowledge_zip=dev_knowledge_zip,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
    client=client,
)

Corpus grande detectado: 63020 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 568
Documentos únicos: 566
Documentos finales: 100
Chunks iniciales: 63020
Chunks finales: 3947


Batches: 100%|██████████| 62/62 [12:05<00:00, 11.70s/it]


In [70]:
print("Claim:")
print(row_305["claim"])

print("\nGold:")
print(normalize_averitec_label(row_305["label"]))

print("\nPredicted:")
print(result_305["verification"].verdict)

print("\nEvidence sufficient:")
print(result_305["verification"].evidence_sufficient)

print("\nExplanation:")
print(result_305["verification"].explanation)

Claim:
Photo Showing the remains of Vladimir Komarov, a man who fell from space, in 1967.

Gold:
SUPPORTED

Predicted:
NOT_ENOUGH_EVIDENCE

Evidence sufficient:
False

Explanation:
The evidence supports the background fact that Vladimir Komarov died in the Soyuz 1 crash in 1967. Several posts also use the exact claimed caption, and one source mentions an open-casket image of his remains. However, none of the provided evidence independently identifies, authenticates, or links the specific photo in the claim to Komarov. Therefore, the claim about what the photo shows cannot be verified from the supplied evidence alone.


In [72]:
for i, evidence in enumerate(result_305["evidences"], start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("URL:", evidence["url"])
    print("Source type:", evidence["source_type"])
    print("Score:", evidence["score"])
    print("Text:", evidence["text"][:1500])


--- EVIDENCE 1 ---
URL: https://www.reddit.com/r/pics/comments/31i4g5/the_remains_of_astronaut_vladimir_komarov_the_man/
Source type: claim+question
Score: 0.8535376787185669
Text: The remains of astronaut Vladimir Komarov, the man who fell from space, 1967. Archived post. New comments cannot be posted and votes cannot be cast. Reddit and its partners use cookies and similar technologies to provide you with a better experience. By accepting all cookies, you agree to our use of cookies to deliver and maintain our services and site, improve the quality of Reddit, personalize Reddit content and advertising, and measure the effectiveness of advertising. By rejecting non-essential cookies, Reddit may still use certain cookies to ensure the proper functionality of our platform. For more information, please see our Cookie Notice and our Privacy Policy.

--- EVIDENCE 2 ---
URL: https://www.reddit.com/r/oldschoolcreepy/comments/haanz9/the_remains_of_the_astronaut_vladimir_komarov_a/
Source typ

##### Claim 305 — `SUPPORTED`: `NOT_ENOUGH_EVIDENCE`

La claim afirma que una fotografía muestra los restos del cosmonauta Vladimir Komarov
tras el accidente de Soyuz 1 en 1967.

El retrieval recupera numerosas fuentes relacionadas directamente con la afirmación.
Varias de ellas utilizan descripciones prácticamente idénticas de la fotografía y otras
documentan tanto la muerte de Komarov en 1967 como la existencia de imágenes de sus
restos tras el accidente.

Sin embargo, el `Evidence Verifier` devuelve `NOT_ENOUGH_EVIDENCE` y establece
`evidence_sufficient=False`, argumentando que ninguna de las evidencias identifica o
autentica de forma independiente la fotografía concreta incluida en la claim.

En este caso, el error pone de manifiesto una limitación adicional del sistema. El
pipeline de AVeriTeC implementado trabaja exclusivamente sobre evidencia textual y no
analiza directamente el contenido visual de las imágenes. Por tanto, en claims cuya
verificación depende de identificar o autenticar una fotografía, el sistema únicamente
puede utilizar captions y descripciones textuales de las fuentes recuperadas.

El retrieval proporciona información textual ampliamente compatible con la etiqueta
`SUPPORTED`, pero el verifier aplica un criterio más restrictivo y exige una
autenticación de la imagen que el pipeline textual no puede realizar directamente. El
error se atribuye principalmente a esta limitación de modalidad, junto con un criterio
conservador del `Evidence Verifier`.

#### Resumen del análisis de errores representativos

La siguiente tabla resume los principales patrones observados en las claims seleccionadas
del conjunto `dev`. El objetivo no es reajustar el sistema a partir de estos casos, sino
identificar de forma descriptiva qué componente del pipeline parece estar relacionado
con cada error.

| Claim | Gold | Predicted | evidence_sufficient | Diagnóstico principal | Interpretación |
|---|---|--|--|---|-------|
| 457 | `CONFLICTING_EVIDENCE` | `REFUTED` | `True` | Evidence Verifier | El retrieval recupera información suficiente sobre componentes apoyados y contradichos, pero el verifier da prioridad al componente refutado y reduce el caso completo a `REFUTED`. |
| 482 | `CONFLICTING_EVIDENCE` | `NOT_ENOUGH_EVIDENCE` | `False` | Mixto, principalmente Evidence Verifier | El retrieval recupera el contexto central y evidencias sobre la tergiversación de la posición de Biden, aunque no la cita exacta completa. El verifier se centra en esa ausencia y no integra adecuadamente los componentes parcialmente respaldados y contradichos. |
| 333 | `REFUTED` | `NOT_ENOUGH_EVIDENCE` | `False` | Retrieval | Las evidencias recuperadas contienen contexto sobre demoliciones y AMMC, pero no la información específica necesaria para refutar la afirmación relativa a las 37 estates. |
| 68 | `NOT_ENOUGH_EVIDENCE` | `REFUTED` | `True` | Evidence Verifier | El retrieval recupera el contexto original y la transcripción relevante, pero el verifier interpreta la ausencia de la cita exacta como una refutación concluyente, alcanzando una conclusión más fuerte de la que permite la evidencia. |
| 57 | `SUPPORTED` | `REFUTED` | `True` | Mixto: retrieval + Evidence Verifier | El retrieval identifica correctamente la localización real de Grand Island, New York, pero no recupera evidencia clara sobre la votación fiscal. El verifier da un peso decisivo al error geográfico y refuta la claim completa. |
| 305 | `SUPPORTED` | `NOT_ENOUGH_EVIDENCE` | `False` | Limitación de modalidad + Evidence Verifier | El retrieval recupera descripciones textuales compatibles con la claim, pero el sistema no analiza directamente la imagen. El verifier exige una autenticación visual que el pipeline textual no puede proporcionar. |

En conjunto, el análisis muestra que los errores no se concentran en un único componente del sistema. Algunos casos se relacionan principalmente con el razonamiento del `Evidence Verifier`, especialmente en claims multicomponente o cuando debe distinguir entre contradicción directa, evidencia parcial y falta de evidencia. Otros errores se
deben a limitaciones del retrieval, que puede no recuperar la información decisiva para la afirmación concreta. También se observa una limitación específica en claims de naturaleza visual, ya que el pipeline implementado utiliza únicamente evidencia textual.

Estos resultados ayudan a delimitar las principales áreas de mejora futura del sistema: mejor descomposición de claims, recuperación más específica por componentes, mecanismos más robustos para distinguir `REFUTED` de `CONFLICTING_EVIDENCE` y soporte multimodal para afirmaciones cuya verificación dependa de imágenes.

### 17. Conclusiones de la evaluación con AVeriTeC

El componente de verificación factual se evaluó sobre una muestra estratificada de 40 claims del conjunto `dev` de AVeriTeC, seleccionando 10 ejemplos de cada una de las cuatro clases. La configuración del sistema se mantuvo congelada durante esta fase y los resultados obtenidos no se utilizaron para realizar nuevos ajustes.

La evaluación obtuvo una `accuracy` de 0.40 y un `macro-F1` de 0.36. El rendimiento fue desigual entre clases. El sistema obtuvo mejores resultados en `SUPPORTED` y `REFUTED`, mientras que presentó dificultades especialmente relevantes en `CONFLICTING_EVIDENCE`, categoría para la que no se clasificó correctamente ninguna de las diez claims evaluadas.

El análisis descriptivo de errores mostró que los fallos no proceden de un único componente. Se observaron errores asociados al `Evidence Verifier`, especialmente en claims multicomponente y en la distinción entre contradicción directa, información parcial y evidencia conflictiva. También se identificaron limitaciones del retrieval cuando la evidencia decisiva no alcanzaba el conjunto final de fragmentos recuperados. Adicionalmente, se observó una limitación de modalidad en claims cuya verificación depende de contenido visual, ya que el pipeline desarrollado
opera exclusivamente sobre evidencia textual.

Durante el análisis posterior también se observó variabilidad entre ejecuciones del componente basado en LLM. Por este motivo, las métricas reportadas corresponden a la ejecución original sobre la muestra reservada de `dev`; las ejecuciones posteriores de claims individuales se utilizaron únicamente con fines diagnósticos.

En conjunto, la evaluación permite validar experimentalmente el funcionamiento del pipeline factual y, al mismo tiempo, delimitar sus principales limitaciones. Los resultados no implican que el sistema resuelva de forma completa la verificación de claims reales, sino que proporcionan una medida reproducible de su comportamiento y
una base para identificar mejoras futuras relacionadas con descomposición de claims, retrieval más específico, razonamiento sobre evidencia conflictiva y soporte multimodal.

#### Configuración resultante del componente factual

Tras finalizar la evaluación, el componente factual se considera cerrado para esta versión del proyecto. Las funciones y configuraciones seleccionadas se trasladarán a módulos Python reutilizables dentro de `src/`, manteniendo sin cambios la lógica evaluada en AVeriTeC.

El notebook se conservará como registro del proceso experimental, mientras que los módulos de `src/` contendrán únicamente las versiones finales necesarias para la integración posterior con el resto de la arquitectura.

### 18. Validación del módulo `retrieval.py`

In [81]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [74]:
from src.retrieval import (
    load_knowledge_records,
    build_claim_index,
    retrieve_evidence,
)

Vamos a probarlas con una claim pequeña, la 93 por ejemplo:

In [75]:
claim_id = 93
claim = dev_df.loc[claim_id, "claim"]

knowledge_records_test = load_knowledge_records(
    knowledge_zip=dev_knowledge_zip,
    claim_id=claim_id,
)

print("Knowledge records:", len(knowledge_records_test))

Knowledge records: 822


Construimos el indice:

In [76]:
chunks_test, index_test, retrieval_info_test = build_claim_index(
    claim=claim,
    knowledge_records=knowledge_records_test,
    embedding_model=embedding_model,
    text_splitter=text_splitter,
)

print(retrieval_info_test)

Corpus grande detectado: 41313 chunks.
Se aplicará TF-IDF para seleccionar 100 documentos.
Documentos originales: 645
Documentos únicos: 645
Documentos finales: 100
Chunks iniciales: 41313
Chunks finales: 1412


Batches: 100%|██████████| 23/23 [02:56<00:00,  7.66s/it]

{'original_documents': 645, 'unique_documents': 645, 'final_documents': 100, 'initial_chunks': 41313, 'final_chunks': 1412, 'tfidf_filter_used': True}


Hacemos el retrieval:

In [77]:
evidences_test = retrieve_evidence(
    claim=claim,
    embedding_model=embedding_model,
    index=index_test,
    chunks=chunks_test,
)

print("Número de evidencias:", len(evidences_test))

Número de evidencias: 10


In [78]:
for i, evidence in enumerate(evidences_test, start=1):
    print(f"\n--- EVIDENCE {i} ---")
    print("Score:", evidence["score"])
    print("URL:", evidence["url"])
    print("Text:", evidence["text"][:500])


--- EVIDENCE 1 ---
Score: 0.8889598846435547
URL: https://news.nd.edu/news/notre-dame-law-school-professor-barrett-nominated-to-us-supreme-court/
Text: Judge Amy Coney Barrett, professor of law at the University of Notre Dame and a 1997 graduate of Notre Dame Law School, was nominated today to the Supreme Court of the United States to fill the vacancy created by the death of Associate Justice Ruth Bader Ginsburg. She is the first Notre Dame graduate and faculty member to be nominated to serve on the nation’s highest court. “The same impressive intellect, character and temperament that made Judge Barrett a successful nominee for the U.S. Court o

--- EVIDENCE 2 ---
Score: 0.8565313816070557
URL: https://swsmmagazine.com/2020/10/working-mom-distinguished-law-professor-and-supreme-court-nominee-a-profile-of-amy-coney-barrett/
Text: On Saturday, September 26, President Trump nominated Judge Amy Coney Barrett to the Supreme Court. Judge Barrett was appointed to the U.S. Court of Appeals fo

Con esto, hemos comprobado que retrieval.py se importa correctamente, lee el Knowledge Store, construye el índice, recupera 10 evidencias

### 19. Validación del módulo `evidence_verifier.py

In [84]:
from src.evidence_verifier import (
    VerificationResult,
    verify_evidence,
)

In [85]:
verification_test = verify_evidence(
    claim=claim,
    evidences=evidences_test,
    client=client,
)

In [86]:
print("Verdict:")
print(verification_test.verdict)

print("\nEvidence sufficient:")
print(verification_test.evidence_sufficient)

print("\nExplanation:")
print(verification_test.explanation)

Verdict:
SUPPORTED

Evidence sufficient:
True

Explanation:
The claim is supported by evidence stating that Barrett graduated summa cum laude and received the Hoynes Prize for the best record in scholarship, deportment, and achievement at Notre Dame Law School. Additional evidence describes the Hoynes Prize as the law school's top honor and quotes a professor calling her its best student.


In [ ]:
#Viendo las relaciones por evidencia
for assessment in verification_test.evidence_assessments:
    print(
        assessment.evidence_id,
        assessment.relation,
        assessment.reason,
    )

1 NEUTRAL It establishes that Barrett graduated from Notre Dame Law School in 1997, but gives no information about her class standing.
2 SUPPORTS It says Barrett graduated as a Kiley Fellow and received the Hoynes Prize, described as the law school's top honor, supporting the claim that she was academically at the top of her class.
3 NEUTRAL It identifies Barrett as a 1997 J.D. alumna but does not address her academic rank or class standing.
4 NEUTRAL The excerpt confirms her connection to Notre Dame Law School but contains no stated information about graduating at the top of her class.
5 SUPPORTS A Notre Dame Law School professor is quoted describing Barrett as the best student and most talented person ever to come through the law school, which supports the asserted top-of-class characterization.
6 SUPPORTS It states that Barrett graduated summa cum laude and received the Hoynes Prize for the best record in scholarship, deportment, and achievement. This directly supports that she grad

Aqui hemos comprobaddo que evidence_verifier.py se importa correctamente, que recibe las evidencias recuperadas, que llama al LLM y devuelve un VerificationResult válido